In [73]:
import os
import warnings
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.mixture import GaussianMixture
from sklearn.tree import DecisionTreeClassifier
from itertools import product as itertools_product

warnings.filterwarnings("ignore")

In [674]:
# ── 1. CONFIG ─────────────────────────────────────────────────────────────────
CONFIG = {
    # ── 파일 경로 ──────────────────────────────────────────────────────────────
    "h5ad_path"          : "/data/yeohs0212/MM/GSE207938/03_ADM_for_trajectory.h5ad",       # ← 실제 경로로 변경

    # pySCENIC 필수 DB ─────────────────────────────────────────────────────────
    "tf_list_path"       : "/data/yeohs0212/scenic/resources/allTFs_hg38.txt",
    "cistarget_db_path"  : [
        # 10kbp upstream/downstream: 원거리 인핸서 포함, 더 넓은 조절 영역
        "/data/yeohs0212/scenic/database/v10/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather",
        # 500bp upstream / 100bp downstream: 프로모터 근위부 특이적
        "/data/yeohs0212/scenic/database/v10/hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather",
    ],
    "motif_annot_path"   : "/data/yeohs0212/scenic/resources/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl",

    # ── AnnData 필드명 ─────────────────────────────────────────────────────────
    # Monocle3 pseudotime: adata.obs가 아닌 별도 CSV 파일에서 로드
    "pseudotime_csv_path"    : "/data/yeohs0212/MM/GSE207938/pseudotime_results.csv",
    "pseudotime_csv_cell_col": "cell_id",   # CSV에서 cell barcode 컬럼명
    #   ※ 실제 CSV 헤더를 확인하고 맞게 수정:
    #     예) "cell", "barcode", "Cell", "CellID" 등
    "pseudotime_csv_val_col" : "PT", # CSV에서 pseudotime 값 컬럼명
    #   ※ 실제 CSV 헤더 확인:
    #     예) "Pseudotime", "pseudotime_value", "pt" 등
    "pseudotime_key"         : "pseudotime",  # adata.obs 에 추가할 컬럼명 (변경 불필요)

    "celltype_key"       : "cell_type",             # ← adata.obs의 실제 컬럼명으로 변경
    "target_celltypes"   : ["ADM", "Acinar", "Bridge", "PDAC"],      # ← trajectory cell type 이름으로 변경

    # raw count 위치 (확인됨: adata.layers['counts'])
    "raw_count_layer"    : "counts",
    
    # Edge sign / Inhibition 검출 파라미터 ───────────────────────────────────
    # SCENIC regulon 내 TF-target AUC 반상관으로 inhibition 검출
    #
    # r_auc < -inhibition_r_threshold  → inhibition
    # r_auc ≥ -inhibition_r_threshold  → activation
    #
    # 0.0 (기본) : r < 0 이면 모두 inhibition  → inhibition 최대 검출
    # 0.1~0.2   : 약한 음의 상관은 activation 으로 유지  → 균형
    # 0.3~      : 강한 음의 상관만 inhibition  → 보수적
    #
    # 실행 후 출력되는 r 분포 히스토그램(tf_tf_r_distribution.pdf)을
    # 확인하고 threshold를 조정하세요.
    "inhibition_r_threshold" : 0.0,
    # ── pySCENIC 파라미터 ──────────────────────────────────────────────────────
    "n_top_genes"        : 3000,
    "num_workers"        : 8,                       # CPU 코어 수에 맞게 조정
    "seed"               : 42,

    # Cell type별 AUC 활성 TF 필터 (Step 2-4) ─────────────────────────────────
    # 적어도 하나의 cell type에서 평균 AUC ≥ 이 값인 TF만 이후 분석에 사용
    # → 어느 cell type에서도 꺼진 TF를 node에서 원천 제거
    # 낮출수록 TF 더 많이 포함 (0.0 = 필터 없음)
    # 높일수록 확실히 활성화되는 TF만 선택
    # 권장 시작값: 0.01 ~ 0.05 (AUC scale에 따라 조정)
    "auc_celltype_min_mean"  : 0.01,
    # 낮출수록 더 많은 TF 포함, 높일수록 cell type-specific TF만 선택
    # 권장: 0.01 ~ 0.05
    "rss_threshold": 0.3,
    # ── Binarization 파라미터 ─────────────────────────────────────────────────
    "min_cells_active"   : 0.01,                    # 최소 1% 세포에서 active인 TF만 유지
    "pseudotime_bins"    : 200,

    # ── Boolean network 파라미터 ──────────────────────────────────────────────
    "max_regulators"     : 7,
    "tree_max_depth"     : 5,
    "min_edge_corr"      : 0.05,                     # TF-TF 최소 |Pearson r|
    
    # QMC sliding window 파라미터 (rule_method="qmc" 일 때 적용)
    #   window_size=1, window_step=1  → 기본 consecutive t→t+1 (QMC 기본값)
    #   window_size↑                 → 노이즈 평균화, 희귀 transition 포착 감소
    #   window_step↓                 → 더 많은 (X,Y) 쌍 생성, 진리표 coverage↑
    "window_size"        : 20,
    "window_step"        : 2,
    
    # Boolean rule 학습 방식 ──────────────────────────────────────────────────
    # "decision_tree" : DecisionTree / REVEAL 방식 (기본, 빠름)
    # "qmc"           : Quine-McCluskey 최소 SOP (느리지만 최소 논리식 보장)
    #                   don't-care 항 활용, regulator 수 ≤ 6 권장 (2^n 복잡도)
    "rule_method"        : "qmc",

    # ── 출력 경로 ─────────────────────────────────────────────────────────────
    "output_dir"         : "./PDAC_boolean_output",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
OUT = CONFIG["output_dir"]


In [675]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1: 데이터 로드 & 전처리 (HVG 계산 스킵 버전)
# ════════════════════════════════════════════════════════════════════════════
def step1_load_and_preprocess(config):
    """
    - AnnData 로드 + 구조 진단 출력
    - Monocle3 pseudotime CSV → adata.obs 에 merge
    - target cell type 선택
    - raw count: adata.layers['counts'] → adata.X 에 세팅
    - [SKIP] HVG 계산 스킵 (이미 HVG만 남은 adata 가정)
    - TF subset 확인 → pySCENIC 입력 준비
    - GRNBoost2용 log1p(CPM) / AUCell용 raw count 분리 보존
    """
    print("\n" + "="*70)
    print("STEP 1: 데이터 로드 & 전처리 (HVG 계산 스킵)")
    print("="*70)

    # ── 1-0. AnnData 로드 + 진단 ────────────────────────────────────────────
    adata = ad.read_h5ad(config["h5ad_path"])
    print(f"  로드된 데이터: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
    # 이미 필터링된 데이터임을 명시적으로 출력
    print("  [알림] 이미 HVG 필터링이 완료된 데이터를 사용합니다.")

    # ── 1-1. Pseudotime CSV → adata.obs merge ───────────────────────────────
    pt_csv_path  = config["pseudotime_csv_path"]
    pt_csv_cell  = config["pseudotime_csv_cell_col"]
    pt_csv_val   = config["pseudotime_csv_val_col"]
    pt_key       = config["pseudotime_key"]

    print(f"  Pseudotime CSV 로드: {pt_csv_path}")
    pt_df = pd.read_csv(pt_csv_path)
    pt_df = pt_df.set_index(pt_csv_cell)[[pt_csv_val]]
    pt_df.columns = [pt_key]

    adata.obs = adata.obs.join(pt_df, how="left")

    # 유효한 Pseudotime 필터링
    pt_series = pd.to_numeric(adata.obs[pt_key], errors="coerce")
    adata.obs[pt_key] = pt_series
    valid_pt  = np.isfinite(adata.obs[pt_key].values.astype(float))
    adata = adata[valid_pt].copy()
    print(f"  Pseudotime 필터링 완료: {adata.shape[0]:,} cells 남음")

    # ── 1-2. Target cell type 선택 ──────────────────────────────────────────
    ct_key = config["celltype_key"]
    targets = config["target_celltypes"]
    mask  = adata.obs[ct_key].isin(targets)
    adata = adata[mask].copy()
    print(f"  Cell type 필터 {targets}: {adata.shape[0]:,} cells 남음")

    # ── 1-3. raw count → adata.X 세팅 ───────────────────────────────────────
    layer_key = config["raw_count_layer"]
    if layer_key in adata.layers:
        adata.X = adata.layers[layer_key].copy()
        print(f"  raw count: adata.layers['{layer_key}'] → adata.X")
    else:
        raise KeyError(f"'{layer_key}'가 adata.layers에 없습니다.")

    # ── 1-4. TF 확인 (HVG 계산은 스킵) ───────────────────────────────────────
    # 이미 adata가 HVG로만 구성되어 있으므로, 현재 adata의 유전자들 자체가 분석 대상입니다.
    # 단, TF 리스트 중 현재 데이터에 살아남은 TF가 무엇인지만 확인합니다.
    tf_list    = pd.read_csv(config["tf_list_path"], header=None)[0].tolist()
    tf_in_data = [tf for tf in tf_list if tf in adata.var_names]
    
    # 이미 HVG로 필터링된 상태이므로 별도의 subset 과정 없이 adata를 그대로 유지합니다.
    adata_s = adata.copy()
    print(f"\n  현재 데이터 유지: {adata_s.shape[1]:,} genes (HVG 이미 적용됨)")
    print(f"  데이터 내 TF 확인: {len(tf_in_data)} TFs found")

    # ── 1-5. GRNBoost2용 log-norm / AUCell용 raw count 분리 ─────────────────
    adata_s.layers["raw_counts"] = adata_s.X.copy() # raw count 백업
    sc.pp.normalize_total(adata_s, target_sum=1e4)
    sc.pp.log1p(adata_s)
    print(f"  adata.X         → log1p(CPM)  [GRNBoost2 입력]")
    print(f"  layers['raw_counts'] → integer count  [AUCell 입력]")

    # ── 1-6. pseudotime 정렬 ────────────────────────────────────────────────
    pt_order     = adata_s.obs[pt_key].argsort()
    adata_sorted = adata_s[pt_order].copy()
    print(f"\n  ✅ Step 1 완료: {adata_sorted.shape[0]:,} cells × "
          f"{adata_sorted.shape[1]:,} genes (pseudotime 정렬 완료)")

    return adata_sorted, tf_in_data

In [676]:
def step2_pyscenic_grn(adata, tf_list, config, ct_series=None):
    """
    pySCENIC 3단계 + RSS 기반 TF 필터:
      2-1. GRNBoost2   → adjacency matrix
      2-2. cisTarget   → regulon pruning
      2-3. AUCell      → cell × TF activity score
      2-4. RSS 기반 필터 → 어느 cell type에서도 특이적이지 않은 TF 제거
    """
    print("\n" + "="*70)
    print("STEP 2: pySCENIC GRN 추론")
    print("="*70)
    import pyscenic
    from arboreto.utils import load_tf_names
    from arboreto.algo import grnboost2
    from pyscenic.prune import prune2df, df2regulons
    from pyscenic.aucell import aucell
    from pyscenic.utils import modules_from_adjacencies
    from pyscenic.rss import regulon_specificity_scores   # RSS import
    import ctxcore.rnkdb as rnkdb

    mat = adata.X.toarray() if hasattr(adata.X, "toarray") else np.array(adata.X)
    expr_df = pd.DataFrame(mat, index=adata.obs_names, columns=adata.var_names)
    print(f"  [표현량] GRNBoost2 입력: log1p-CPM  {expr_df.shape}")

    if "raw_counts" in adata.layers:
        raw_mat = adata.layers["raw_counts"]
        raw_mat = raw_mat.toarray() if hasattr(raw_mat, "toarray") else np.array(raw_mat)
        expr_df_raw = pd.DataFrame(raw_mat, index=adata.obs_names, columns=adata.var_names)
        print(f"  [표현량] AUCell 입력:    raw count   {expr_df_raw.shape}")
    else:
        expr_df_raw = expr_df
        print("  ⚠ raw_counts 레이어 없음. AUCell에 log-norm 사용 (fallback)")

    # ── 2-1. GRNBoost2 ───────────────────────────────────────────────────────
    print("\n  [2-1] GRNBoost2 실행 중... (10~60분 소요)")
    adjacencies = grnboost2(
        expression_data=expr_df,
        tf_names=tf_list,
        verbose=True,
        seed=config["seed"]
    )
    adjacencies.to_csv(f"{OUT}/adjacencies.csv", index=False)
    print(f"       → {len(adjacencies)} TF-gene links 저장")

    # ── 2-2. cisTarget ───────────────────────────────────────────────────────
    print("\n  [2-2] cisTarget regulon pruning 중... (DB 2개 병렬 사용)")
    dbs = [
        rnkdb.FeatherRankingDatabase(fname=db, name=os.path.basename(db))
        for db in config["cistarget_db_path"]
    ]
    for db in dbs:
        print(f"         • {db.name}")

    modules = list(modules_from_adjacencies(adjacencies, expr_df))
    df_motifs = prune2df(
        dbs,
        modules=modules,
        motif_annotations_fname=config["motif_annot_path"],
        client_or_address="custom_multiprocessing",
        num_workers=config["num_workers"]
    )
    regulons = df2regulons(df_motifs)
    print(f"       → {len(regulons)} regulons 추론됨")

    regulon_dict = {r.name.split("(")[0]: set(r.genes) for r in regulons}

    # ── 2-3. AUCell ──────────────────────────────────────────────────────────
    print("\n  [2-3] AUCell TF activity 계산 중...")
    auc_mtx = aucell(
        expr_df_raw,
        signatures=regulons,
        num_workers=config["num_workers"],
        seed = config["seed"]
    )
    auc_mtx.columns = [c.split("(")[0] for c in auc_mtx.columns]
    auc_mtx.to_csv(f"{OUT}/auc_matrix_all.csv")
    print(f"       → AUC matrix (필터 전): {auc_mtx.shape}")

    # ── 2-4. RSS 기반 TF 필터 ────────────────────────────────────────────────
    # RSS (Regulon Specificity Score): 각 TF regulon이 각 cell type에 얼마나
    # 특이적으로 활성화되는지를 Jensen-Shannon divergence 기반으로 계산.
    # → max(cell type별 RSS) >= rss_threshold 인 TF만 유지
    # → 어느 cell type에서도 특이적이지 않은 TF = 배경 노이즈 제거
    #
    # 단순 평균 AUC 필터 대비 장점:
    #   - 높은 평균 AUC여도 모든 cell type에 고르게 발현되면 낮은 RSS
    #     (→ transition에 특이적이지 않으므로 제거)
    #   - 낮은 평균 AUC여도 특정 cell type에서만 강하게 발현되면 높은 RSS
    #     (→ transition-specific TF이므로 유지)
    
    def regulon_specificity_scores_real(auc_mtx, cell_type_series):

        from math import sqrt

        import numpy as np
        import pandas as pd
        from scipy.spatial.distance import jensenshannon
        cell_types = list(cell_type_series.unique())
        n_types = len(cell_types)
        regulons = list(auc_mtx.columns)
        n_regulons = len(regulons)
        rss_values = np.empty(shape=(n_types, n_regulons), dtype=float)
        def rss(aucs, labels):
            # jensenshannon function provides distance which is the sqrt of the JS divergence.
            return 1.0 - jensenshannon(aucs / aucs.sum(), labels / labels.sum())

        for cidx, regulon_name in enumerate(regulons):
            for ridx, cell_type in enumerate(cell_types):
                rss_values[ridx, cidx] = rss(
                    auc_mtx[regulon_name], (cell_type_series == cell_type).astype(int)
                )

        return pd.DataFrame(data=rss_values, index=cell_types, columns=regulons)

    if ct_series is not None and len(ct_series) > 0:
        print("\n  [2-4] RSS 기반 TF 필터")

        # AUCell 결과와 cell type 정보 정렬
        common        = auc_mtx.index.intersection(ct_series.index)
        auc_aligned   = auc_mtx.loc[common]
        ct_aligned    = ct_series.loc[common]

        # RSS 계산: 결과 shape = cell_type × TF
        rss_df = regulon_specificity_scores_real(auc_aligned, ct_aligned)
        rss_df.to_csv(f"{OUT}/rss_scores.csv")
        print(f"         RSS matrix: {rss_df.shape}  → {OUT}/rss_scores.csv")

        # cell type별 max RSS 출력
        rss_threshold = config.get("rss_threshold", 0.01)
        max_rss       = rss_df.max(axis=0)   # TF별 최대 RSS
        keep_tfs      = max_rss[max_rss >= rss_threshold].index.tolist()

        n_before = auc_mtx.shape[1]
        auc_mtx      = auc_mtx[keep_tfs]
        regulon_dict = {tf: genes for tf, genes in regulon_dict.items()
                        if tf in keep_tfs}

        print(f"         rss_threshold: {rss_threshold}")
        print(f"         TF: {n_before} → {len(keep_tfs)} 유지 "
              f"({n_before - len(keep_tfs)} 제거)")

        # cell type별 top RSS TF 출력
        print(f"\n         Cell type별 top-5 RSS TF:")
        for ct in rss_df.index:
            top5 = rss_df.loc[ct].sort_values(ascending=False).head(5)
            print(f"           {ct:<20}: {top5.index.tolist()}")

        # RSS 히트맵 저장
        fig, ax = plt.subplots(figsize=(max(8, len(keep_tfs)*0.3),
                                         max(3, len(rss_df)*0.5)))
        sns.heatmap(
            rss_df[keep_tfs],
            ax=ax, cmap="YlOrRd",
            linewidths=0.3, linecolor="#eee",
            cbar_kws={"label": "RSS", "shrink": 0.7}
        )
        ax.set_title("Regulon Specificity Scores (RSS) per Cell Type",
                     fontsize=11, fontweight="bold")
        ax.set_xlabel("TF regulon", fontsize=9)
        ax.set_ylabel("Cell type", fontsize=9)
        ax.tick_params(axis="x", rotation=90, labelsize=7)
        plt.tight_layout()
        plt.savefig(f"{OUT}/rss_heatmap.pdf", dpi=150, bbox_inches="tight")
        plt.savefig(f"{OUT}/rss_heatmap.png", dpi=150, bbox_inches="tight")
        plt.close()
        print(f"         RSS 히트맵 저장: {OUT}/rss_heatmap.pdf/png")
    else:
        print("\n  [2-4] ct_series 없음 — RSS 필터 스킵")

    auc_mtx.to_csv(f"{OUT}/auc_matrix.csv")
    print(f"\n       → 최종 AUC matrix: {auc_mtx.shape}")

    return adjacencies, regulon_dict, auc_mtx

In [677]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 3: TF-TF 네트워크 추출 + Edge sign (activation / inhibition)
# ════════════════════════════════════════════════════════════════════════════
#
# 전략: SCENIC regulon 내 TF-target 반상관 세트로 inhibition 검출
# ─────────────────────────────────────────────────────────────────────────
# pySCENIC은 TF가 target을 "activation"한다고 가정하고 regulon을 구성한다.
# 하지만 GRNBoost2는 방향성(activation/inhibition)을 구분하지 않고
# importance score만 출력하므로, regulon 내에 실제로 억제 타겟이 섞여 있다.
#
# 핵심 아이디어:
#   regulator A의 regulon에 target B가 포함되어 있을 때,
#   A의 AUC activity와 B의 AUC activity 사이의 Pearson correlation을
#   pairwise로 계산한다.
#
#   Pearson r(A_auc, B_auc) < -threshold  → A가 B를 억제 (inhibition)
#   Pearson r(A_auc, B_auc) ≥ -threshold  → A가 B를 활성화 (activation)
#
# 이 방식이 단순 global AUC corr와 다른 이유:
#   - 엣지 후보가 이미 regulon에 포함된 pair로 제한되어 있음
#     (GRNBoost2가 importance > 0이라고 판단한 직접 조절 관계)
#   - 따라서 "B가 A의 regulon target"이라는 구조적 증거 + AUC corr 음수
#     = direct repressor 관계의 강한 증거
#   - 기존에 inhibition이 거의 없었던 이유는 regulon이 대부분 양의 패턴으로
#     pruning되었기 때문 → 음의 corr를 보이는 타겟은 실제로 억제 증거
#
# 추가로 adjacencies (GRNBoost2 raw)에서 importance score를 엣지 가중치로 활용.
# ════════════════════════════════════════════════════════════════════════════

def _build_regulon_auc_corr_table(regulon_dict, auc_mtx, tf_set):
    """
    regulon_dict의 모든 TF→target 쌍 중 target이 TF인 경우에 대해
    AUC pairwise correlation을 계산하여 DataFrame으로 반환.

    반환 컬럼: source, target, r_auc, pval_auc
    """
    tf_set  = set(tf_set)
    auc_tfs = set(auc_mtx.columns)
    rows    = []

    for reg, targets in regulon_dict.items():
        if reg not in auc_tfs:
            continue
        # target 중 TF인 것만 (TF→TF 엣지)
        tf_targets = targets.intersection(tf_set).intersection(auc_tfs)
        for tgt in tf_targets:
            if reg == tgt:
                continue
            r, p = pearsonr(auc_mtx[reg].values, auc_mtx[tgt].values)
            rows.append({
                "source" : reg,
                "target" : tgt,
                "r_auc"  : round(float(r), 4),
                "pval"   : round(float(p), 6),
            })

    return pd.DataFrame(rows)


def step3_extract_tf_tf_network(regulon_dict, auc_mtx, tf_set, config,
                                 expr_df=None, pt_series=None):
    """
    SCENIC regulon 내 TF-target 반상관(negative AUC correlation)으로
    inhibition edge를 검출하고 TF-TF Boolean network edge list를 생성.

    Parameters
    ----------
    regulon_dict : {TF: set(target_genes)}  — pySCENIC step2 결과
    auc_mtx      : cell × TF AUC activity score DataFrame
    tf_set       : 분석 대상 TF 목록 (list)
    config       : CONFIG dict
    expr_df      : 미사용 (하위 호환성 유지)
    pt_series    : 미사용 (하위 호환성 유지)

    Sign 결정 규칙
    ──────────────
    r_auc  < -inhibition_r_threshold  → inhibition (-1)
    r_auc  ≥ -inhibition_r_threshold  → activation (+1)

    min_edge_corr (절댓값 기준) 이하인 엣지는 제거.
    """
    print("\n" + "="*70)
    print("STEP 3: TF-TF 네트워크 추출  (regulon 내 AUC 반상관 기반)")
    print("="*70)

    inh_r_thr   = config.get("inhibition_r_threshold", 0.0)
    min_corr    = config["min_edge_corr"]

    print(f"  inhibition_r_threshold : r_auc < -{inh_r_thr:.2f}  → inhibition")
    print(f"  min_edge_corr          : |r_auc| ≥ {min_corr}")
    print(f"  엣지 후보 범위         : regulon에 포함된 TF→TF pair만")

    # ── 3-1. regulon 내 TF→TF AUC correlation 전량 계산 ──────────────────
    corr_df = _build_regulon_auc_corr_table(regulon_dict, auc_mtx, tf_set)
    print(f"\n  regulon 내 TF→TF pair: {len(corr_df):,}개 (필터 전)")

    # ── 3-2. |r_auc| < min_edge_corr 제거 ──────────────────────────────────
    corr_df = corr_df[corr_df["r_auc"].abs() >= min_corr].copy()
    print(f"  |r_auc| ≥ {min_corr} 필터 후: {len(corr_df):,}개")

    # ── 3-3. sign 결정 ──────────────────────────────────────────────────────
    # r_auc < -inh_r_thr → inhibition
    # inh_r_thr = 0.0 (기본): r_auc < 0 이면 모두 inhibition
    #             0.0보다 크게 설정하면 더 보수적 (r < -0.2 등)
    corr_df["sign"] = corr_df["r_auc"].apply(
        lambda r: -1 if r < -inh_r_thr else 1
    )
    corr_df["type"] = corr_df["sign"].map({1: "activation", -1: "inhibition"})

    # ── 3-4. 중복 엣지 제거 (같은 source→target 쌍이 복수 regulon에 있는 경우
    #         |r_auc| 가장 큰 것을 유지) ──────────────────────────────────────
    corr_df["abs_r"] = corr_df["r_auc"].abs()
    tf_tf_net = (corr_df
                 .sort_values("abs_r", ascending=False)
                 .drop_duplicates(subset=["source", "target"])
                 .drop(columns=["abs_r"])
                 .reset_index(drop=True))

    tf_tf_net.to_csv(f"{OUT}/tf_tf_network.csv", index=False)

    # ── 통계 출력 ────────────────────────────────────────────────────────────
    n_act  = (tf_tf_net["sign"] ==  1).sum()
    n_inh  = (tf_tf_net["sign"] == -1).sum()
    ratio  = n_inh / len(tf_tf_net) * 100 if len(tf_tf_net) else 0

    # r_auc 분포
    r_neg  = (corr_df["r_auc"] < 0).sum()   # 필터 전 음수 pair 수
    r_pos  = (corr_df["r_auc"] > 0).sum()

    print(f"\n  r_auc 분포 (|r| ≥ {min_corr} 기준):")
    print(f"    r < 0  : {r_neg:,}  → inhibition 후보")
    print(f"    r > 0  : {r_pos:,}  → activation 후보")
    print(f"\n  최종 TF-TF edges : {len(tf_tf_net):,}")
    print(f"    Activation (+)  : {n_act:,}  ({100-ratio:.1f}%)")
    print(f"    Inhibition (−)  : {n_inh:,}  ({ratio:.1f}%)")

    if n_inh == 0:
        print("\n  ⚠ inhibition이 0개입니다.")
        print(f"     → inhibition_r_threshold 를 높여보세요 (현재 {inh_r_thr})")
        print(f"       예: 0.0 → 그대로 (r < 0 이면 inhibition)")
        print(f"       또는 min_edge_corr 를 낮춰보세요 (현재 {min_corr})")
        print(f"     → r_auc 분포 확인: {OUT}/tf_tf_network.csv 에서 r_auc 컬럼 참고")

    # r 분포 히스토그램 저장 (진단용)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(tf_tf_net["r_auc"], bins=40, color="#3498DB", edgecolor="white", linewidth=0.5)
    ax.axvline(-inh_r_thr, color="#E74C3C", linestyle="--", linewidth=1.5,
               label=f"inhibition threshold (r = -{inh_r_thr})")
    ax.axvline(0, color="#888", linestyle=":", linewidth=1)
    ax.set_xlabel("Pearson r  (AUC_regulator  vs  AUC_target)", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title("TF-TF AUC correlation distribution\n(regulon-contained pairs)", fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{OUT}/tf_tf_r_distribution.pdf", dpi=150)
    plt.savefig(f"{OUT}/tf_tf_r_distribution.png", dpi=150)
    plt.close()
    print(f"\n  → r 분포 히스토그램 저장: tf_tf_r_distribution.pdf/png")

    G = nx.DiGraph()
    for _, row in tf_tf_net.iterrows():
        #G.add_edge(row["source"], row["target"],
                   #sign=row["sign"], corr=row["r_auc"], type=row["type"])
        G.add_edge(row["source"], row["target"],
                   sign=row["sign"], corr=row["r_auc"], type=row["type"])
    print(f"  Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    return tf_tf_net, G


In [678]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 4: Pseudotime 기반 TF Binarization
# ════════════════════════════════════════════════════════════════════════════
def step4_binarize_tf_activity(auc_mtx, adata, config):
    """
    각 TF의 AUC score를 GMM 2-component로 0/1 binarize.
    pseudotime에 따라 정렬하여 state transition matrix 생성.
    """
    print("\n" + "="*70)
    print("STEP 4: TF Activity Binarization (pseudotime 기반)")
    print("="*70)
 
    pt_key  = config["pseudotime_key"]
    pt_vals = adata.obs[pt_key].values
 
    # ── 4-1. GMM binarization ───────────────────────────────────────────────
    binary_dict    = {}
    gmm_stats      = []
    gmm_thresholds = {}   # {TF: float}  — Step 4-2에서 재사용할 threshold
 
    for tf in auc_mtx.columns:
        values = auc_mtx[tf].values.reshape(-1, 1)
 
        gmm = GaussianMixture(n_components=2, covariance_type="full",
                               random_state=config["seed"], max_iter=200)
        gmm.fit(values)
        labels = gmm.predict(values)
 
        # active component = 높은 평균값 component
        high_comp  = int(np.argmax(gmm.means_.flatten()))
        low_comp   = 1 - high_comp
        binary     = (labels == high_comp).astype(int)
 
        # threshold = 두 Gaussian 평균의 중간값
        mean_high  = float(gmm.means_[high_comp][0])
        mean_low   = float(gmm.means_[low_comp][0])
        threshold  = (mean_high + mean_low) / 2.0
 
        # 너무 희박한 TF 제거
        frac_active = binary.mean()
        if frac_active < config["min_cells_active"] or frac_active > (1 - config["min_cells_active"]):
            continue
 
        binary_dict[tf]     = binary
        gmm_thresholds[tf]  = threshold
        gmm_stats.append({
            "TF"         : tf,
            "frac_active": round(frac_active, 4),
            "mean_low"   : round(mean_low,  6),
            "mean_high"  : round(mean_high, 6),
            "threshold"  : round(threshold, 6),   # AUC > threshold → 1
        })
 
    binary_df = pd.DataFrame(binary_dict, index=auc_mtx.index)
    pd.DataFrame(gmm_stats).to_csv(f"{OUT}/gmm_stats.csv", index=False)
    print(f"  Binarized TFs: {binary_df.shape[1]} (원본 {auc_mtx.shape[1]}개 에서 min_cells_acitve에 의해 필터링 됨)")
 
    # ── 4-2. Pseudotime 정렬 ───────────────────────────────────────────────
    # auc_mtx 인덱스 순서와 adata 인덱스 정렬
    common_cells = binary_df.index.intersection(adata.obs_names)
    binary_df = binary_df.loc[common_cells]
    pt_series = adata.obs.loc[common_cells, pt_key]
    binary_df["pseudotime"] = pt_series.values
    binary_sorted = binary_df.sort_values("pseudotime").reset_index(drop=True)
    binary_sorted.to_csv(f"{OUT}/binary_tf_pseudotime.csv", index=False)
 
    # ── 4-3. Pseudotime bin별 TF state 시각화 ──────────────────────────────
    tf_cols = [c for c in binary_sorted.columns if c != "pseudotime"]
    n_bins  = config["pseudotime_bins"]
    binary_sorted["pt_bin"] = pd.cut(binary_sorted["pseudotime"], bins=n_bins, labels=False)
    bin_mean = binary_sorted.groupby("pt_bin")[tf_cols].mean()
 
    # TF 시각화
    tf_var = bin_mean.var().sort_values(ascending=False)
    top_tfs = tf_var.index.tolist()
 
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(
        bin_mean[top_tfs].T,
        cmap="RdBu_r", center=0.5, vmin=0, vmax=1,
        ax=ax, linewidths=0.3, cbar_kws={"label": "Fraction active"}
    )
    ax.set_xlabel("Pseudotime bin (Acinar → PDAC)", fontsize=12)
    ax.set_ylabel("TF", fontsize=12)
    ax.set_title("TF Activity Along PDAC Transition Trajectory", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/tf_activity_heatmap.pdf", dpi=150)
    plt.savefig(f"{OUT}/tf_activity_heatmap.png", dpi=150)
    plt.close()
    print(f"  → 시각화 저장: tf_activity_heatmap.pdf/png")
 
    return binary_sorted, tf_cols, gmm_thresholds

In [679]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 4-2: Cell state별 / Cell별 binary TF state 계산
# ════════════════════════════════════════════════════════════════════════════
def step4_2_binary_state_profiles(auc_mtx, adata, tf_tf_net,
                                   gmm_thresholds, config):
    """
    Step 4에서 학습한 GMM threshold를 그대로 재사용하여:
 
    (A) Cell별 binary matrix
        - 모든 cell × (boolean network에 속하는 TF) 의 0/1 값
        - AUC > GMM threshold → 1, 이하 → 0
        - 출력: cell_binary_state.csv  (index=cell barcode)
 
    (B) Cell state별 대표 binary state
        - adata.obs[celltype_key] 기준으로 그룹핑
        - 각 그룹 내 majority vote (≥ 0.5 → 1, < 0.5 → 0)
        - 출력: cellstate_binary_profile.csv  (index=cell type)
                cellstate_binary_heatmap.pdf
 
    Parameters
    ----------
    auc_mtx        : cell × TF AUC DataFrame  (Step 2 결과)
    adata          : AnnData  (obs에 celltype_key 포함)
    tf_tf_net      : TF-TF edge DataFrame  (Step 3 결과)
                     → boolean network에 속하는 TF 목록 추출에 사용
    gmm_thresholds : {TF: threshold}  (Step 4 GMM 결과)
    config         : CONFIG dict
    """
    print("\n" + "="*70)
    print("STEP 4-2: Cell state별 / Cell별 binary TF state 계산")
    print("="*70)
 
    ct_key = config["celltype_key"]
 
    # ── 대상 TF: boolean network에 속하는 노드만 ──────────────────────────────
    # tf_tf_net의 source/target에 등장한 TF = network node
    net_tfs = sorted(
        set(tf_tf_net["source"].tolist() + tf_tf_net["target"].tolist())
    )
    # gmm_thresholds가 있는 TF만 (필터에서 살아남은 TF)
    net_tfs = [tf for tf in net_tfs if tf in gmm_thresholds and tf in auc_mtx.columns]
    print(f"  Boolean network TF 수: {len(net_tfs)}")
 
    # ── (A) Cell별 binary matrix ─────────────────────────────────────────────
    # AUC > GMM threshold → 1
    cell_binary = pd.DataFrame(index=auc_mtx.index, columns=net_tfs, dtype=int)
    for tf in net_tfs:
        thr = gmm_thresholds[tf]
        cell_binary[tf] = (auc_mtx[tf].values > thr).astype(int)
 
    # cell type 정보 추가
    common_cells = cell_binary.index.intersection(adata.obs_names)
    cell_binary  = cell_binary.loc[common_cells].copy()
 
    if ct_key in adata.obs.columns:
        cell_binary.insert(0, "cell_type",
                           adata.obs.loc[common_cells, ct_key].values)
    if config["pseudotime_key"] in adata.obs.columns:
        cell_binary.insert(1, "pseudotime",
                           adata.obs.loc[common_cells, config["pseudotime_key"]].values)
 
    cell_binary.index.name = "cell_id"
    cell_binary.to_csv(f"{OUT}/cell_binary_state.csv")
    print(f"  Cell별 binary matrix → cell_binary_state.csv")
 
    # ── (B) Cell state별 대표 binary state ───────────────────────────────────
    if ct_key not in adata.obs.columns:
        print(f"  ⚠ '{ct_key}' 컬럼 없음 — cell state 프로파일 스킵")
        return cell_binary, None
 
    tf_only_cols = [c for c in cell_binary.columns
                    if c not in ("cell_type", "pseudotime")]
 
    # majority vote: 각 cell state 내 TF별 active cell 비율 계산
    binary_with_ct = cell_binary.copy()
    frac_df = (binary_with_ct
               .groupby("cell_type")[tf_only_cols]
               .mean())           # fraction active per cell type
 
    # majority vote: 0.5 기준 이진화
    majority_df = (frac_df >= 0.5).astype(int)
 
    frac_df.to_csv(f"{OUT}/cellstate_active_fraction.csv")
    majority_df.to_csv(f"{OUT}/cellstate_binary_profile.csv")
    print(f"  Cell state 수: {len(frac_df)}  → cellstate_binary_profile.csv")
    print(f"\n  Cell state별 대표 binary profile (majority vote, 0.5 기준):")
    print(majority_df.to_string())
 
    # ── (C) 시각화 ────────────────────────────────────────────────────────────
    _plot_cellstate_binary_heatmap(frac_df, majority_df, net_tfs)
    _plot_cell_binary_umap(cell_binary, adata, net_tfs, config)
 
    return cell_binary, majority_df
 
def _plot_cellstate_binary_heatmap(frac_df, majority_df, net_tfs):
    """
    Cell state × TF 히트맵 2종:
      (1) fraction active  (연속값, 0~1)
      (2) majority vote    (이진값, 0/1)
    """
    # TF를 pseudotime 축 따라 순서 정렬 (frac_df 행 순서 = cell state 순서 유지)
    n_tfs    = len(net_tfs)
    fig_w    = max(10, n_tfs * 0.35)
    fig_h    = max(3,  len(frac_df) * 0.6)
 
    fig, axes = plt.subplots(1, 2, figsize=(fig_w * 2 + 1, fig_h),
                              gridspec_kw={"wspace": 0.05})
 
    # (1) fraction active
    sns.heatmap(
        frac_df[net_tfs],
        ax=axes[0], cmap="Blues", vmin=0, vmax=1,
        linewidths=0.4, linecolor="#ddd",
        cbar_kws={"label": "Fraction active", "shrink": 0.7},
        annot=(n_tfs <= 30), fmt=".2f", annot_kws={"size": 7}
    )
    axes[0].set_title("Fraction active per cell state", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("TF (boolean network node)", fontsize=9)
    axes[0].set_ylabel("Cell state", fontsize=9)
    axes[0].tick_params(axis="x", rotation=90, labelsize=7)
 
    # (2) majority vote binary
    sns.heatmap(
        majority_df[net_tfs],
        ax=axes[1], cmap="RdBu_r", vmin=0, vmax=1,
        linewidths=0.4, linecolor="#ddd",
        cbar_kws={"label": "Binary state (0/1)", "shrink": 0.7},
        annot=True, fmt="d", annot_kws={"size": 7}
    )
    axes[1].set_title("Majority vote binary state per cell state", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("TF (boolean network node)", fontsize=9)
    axes[1].set_ylabel("")
    axes[1].tick_params(axis="x", rotation=90, labelsize=7)
    axes[1].set_yticklabels([])
 
    plt.suptitle("Boolean Network TF Binary State by Cell State",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.savefig(f"{OUT}/cellstate_binary_heatmap.pdf",
                dpi=150, bbox_inches="tight")
    plt.savefig(f"{OUT}/cellstate_binary_heatmap.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → 히트맵 저장: cellstate_binary_heatmap.pdf/png")

 
 
def _plot_cell_binary_umap(cell_binary, adata, net_tfs, config):
    """
    UMAP 좌표가 adata.obsm['X_umap']에 있으면,
    각 TF의 cell별 binary 값을 UMAP 위에 overlaid scatter로 저장.
    네트워크 TF가 많으면 variance가 높은 상위 9개만 시각화.
    """
    if "X_umap" not in adata.obsm:
        print("  ⚠ adata.obsm['X_umap'] 없음 — UMAP binary 시각화 스킵")
        return
 
    common = cell_binary.index.intersection(adata.obs_names)
    umap_coords = adata.obsm["X_umap"]
    umap_df     = pd.DataFrame(
        umap_coords[adata.obs_names.get_indexer(common)],
        index=common, columns=["UMAP1", "UMAP2"]
    )
 
    tf_only = [c for c in net_tfs
               if c in cell_binary.columns and c not in ("cell_type", "pseudotime")]
 
    # variance 높은 상위 9개 TF 선택
    var_per_tf  = cell_binary[tf_only].var()
    plot_tfs    = var_per_tf.sort_values(ascending=False).head(9).index.tolist()
 
    n_cols = 3
    n_rows = int(np.ceil(len(plot_tfs) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 4, n_rows * 3.5))
    axes = np.array(axes).flatten()
 
    for ax, tf in zip(axes, plot_tfs):
        vals   = cell_binary.loc[common, tf].values.astype(float)
        sc     = ax.scatter(
            umap_df["UMAP1"], umap_df["UMAP2"],
            c=vals, cmap="RdBu_r", vmin=0, vmax=1,
            s=2, alpha=0.7, rasterized=True
        )
        ax.set_title(tf, fontsize=10, fontweight="bold")
        ax.set_xlabel("UMAP1", fontsize=7)
        ax.set_ylabel("UMAP2", fontsize=7)
        ax.tick_params(labelsize=6)
        plt.colorbar(sc, ax=ax, shrink=0.7).set_label("Binary (0/1)", size=7)
 
    for ax in axes[len(plot_tfs):]:
        ax.axis("off")
 
    plt.suptitle("Cell-level Binary TF State on UMAP (top 9 by variance)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUT}/cell_binary_umap.pdf",  dpi=150, bbox_inches="tight")
    plt.savefig(f"{OUT}/cell_binary_umap.png",  dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → UMAP 시각화 저장: cell_binary_umap.pdf/png  (TF: {plot_tfs})")

In [680]:
# ════════════════════════════════════════════════════════════════════════════
# [교체] STEP 5 공통 유틸 (Cell 7) — self-loop 원인 구분 + 상수 rule 허용
# ════════════════════════════════════════════════════════════════════════════

# self-loop 원인 분류
SELFLOOP_REASON = {}   # {TF: "no_inedge" | "constant_rule"}

def _select_regulators(target_tf, tf_cols, tf_tf_net, config):
    """
    tf_tf_net에서 upstream regulator를 선택.

    self-loop가 걸리는 유일한 조건:
      tf_tf_net에 해당 TF를 타겟으로 하는 in-edge가 없을 때
      → SELFLOOP_REASON[target_tf] = "no_inedge"

    제거된 조건 (이전 버전 대비):
      - frac_active < threshold 일 때 self-loop 추가 (제거)
      - 상수 rule → self-loop 대체 (_ensure_nontrivial 제거)
        대신 상수 rule은 "0"/"1" 그대로 허용하되 SELFLOOP_REASON에 기록

    반환: (regulators: list[str], is_selfloop: bool)
    """
    corr_col = "r_auc" if "r_auc" in tf_tf_net.columns else (
               "corr_auc" if "corr_auc" in tf_tf_net.columns else None)
    rows       = tf_tf_net[tf_tf_net["target"] == target_tf]
    rows       = rows[rows["source"].isin(tf_cols)]
    regulators = rows["source"].tolist()

    if len(regulators) > config["max_regulators"]:
        if corr_col:
            corr_vals  = rows.set_index("source")[corr_col].abs()
            regulators = (corr_vals.sort_values(ascending=False)
                                   .head(config["max_regulators"])
                                   .index.tolist())
        else:
            regulators = regulators[: config["max_regulators"]]

    if not regulators:
        SELFLOOP_REASON[target_tf] = "no_inedge"
        return [target_tf], True

    return regulators, False


# ════════════════════════════════════════════════════════════════════════════
# STEP 5-1: Boolean Rule 학습 — Decision Tree (REVEAL 방식)
# ════════════════════════════════════════════════════════════════════════════
def step5_1_learn_rules_decision_tree(binary_sorted, tf_tf_net, config):
    """
    t → t+1 transition 을 DecisionTree 로 학습하여 Boolean expression 도출.

    개선 사항:
      - regulator 없는 TF → self-loop 보장
      - 학습 결과가 상수(0/1) → self-loop 로 대체
      - 모든 TF 가 최소 1개 이상의 regulator 를 가짐
    """
    print("\n" + "="*70)
    print("STEP 5-1: Boolean Rule 학습 (Decision Tree / REVEAL)")
    print("="*70)

    tf_cols     = [c for c in binary_sorted.columns
                   if c not in ("pseudotime", "pt_bin")]
    binary_data = binary_sorted[tf_cols].dropna().astype(int)
    X_all       = binary_data.iloc[:-1].values   # t
    Y_all       = binary_data.iloc[1:].values    # t+1

    rules_dict    = {}
    accuracy_dict = {}
    selfloop_tfs  = []
    replaced_tfs  = []

    for i, target_tf in enumerate(tf_cols):
        regulators, used_sl = _select_regulators(
            target_tf, tf_cols, tf_tf_net, config
        )
        if used_sl:
            # pure self-loop: identity rule
            rules_dict[target_tf]    = rule_str
            accuracy_dict[target_tf] = round(acc, 4)
            continue

        y_idx       = tf_cols.index(target_tf)
        y           = Y_all[:, y_idx]
        reg_indices = [tf_cols.index(r) for r in regulators]
        X           = X_all[:, reg_indices]

        clf = DecisionTreeClassifier(
            max_depth        = config["tree_max_depth"],
            min_samples_leaf = max(5, int(len(X) * 0.01)),
            random_state     = config["seed"],
            class_weight     = "balanced",   # 불균형 클래스 보정
        )
        clf.fit(X, y)
        acc       = clf.score(X, y)
        rule_str  = tree_to_boolean_expr(clf, regulators)

        # 상수 rule 대체
        rule_str, was_replaced = _ensure_nontrivial_rule(rule_str, target_tf, regulators)
        if was_replaced:
            replaced_tfs.append(target_tf)

        rules_dict[target_tf]    = rule_str
        accuracy_dict[target_tf] = round(acc, 4)

        if i % 10 == 0:
            print(f"  [{i+1:>4}/{len(tf_cols)}] {target_tf:<20} "
                  f"acc={acc:.3f}  regs={regulators}")

    accs = [v for v in accuracy_dict.values() if v is not None]
    print(f"\n  평균 accuracy  : {np.mean(accs):.4f}")
    print(f"  중앙값 accuracy: {np.median(accs):.4f}")
    print(f"  Self-loop 적용 : {len(selfloop_tfs)} TFs  {selfloop_tfs[:5]}{'…' if len(selfloop_tfs)>5 else ''}")
    print(f"  상수→self-loop  : {len(replaced_tfs)} TFs  {replaced_tfs[:5]}{'…' if len(replaced_tfs)>5 else ''}")

    return rules_dict, accuracy_dict


def tree_to_boolean_expr(clf, feature_names):
    """
    sklearn DecisionTree → .bnet 호환 Boolean expression 변환.
    (.bnet 문법: & AND,  | OR,  ! NOT,  1 True,  0 False)
    """
    tree  = clf.tree_
    left  = tree.children_left
    right = tree.children_right
    feat  = tree.feature
    value = tree.value

    def recurse(node):
        if left[node] == right[node]:       # leaf
            counts = value[node][0]
            return "1" if counts[1] >= counts[0] else "0"

        fname      = feature_names[feat[node]]
        left_expr  = recurse(left[node])    # feature <= 0.5  → feature == 0
        right_expr = recurse(right[node])   # feature >  0.5  → feature == 1

        if left_expr == right_expr:
            return left_expr

        not_f = f"!{fname}"
        pos_f = fname

        if   left_expr == "0" and right_expr == "1": return pos_f
        elif left_expr == "1" and right_expr == "0": return not_f
        elif left_expr == "0":  return f"({pos_f} & {right_expr})"
        elif right_expr == "0": return f"({not_f} & {left_expr})"
        elif left_expr == "1":  return f"({not_f} | {right_expr})"
        elif right_expr == "1": return f"({pos_f} | {left_expr})"
        else:
            return f"(({not_f} & {left_expr}) | ({pos_f} & {right_expr}))"

    return recurse(0)


# ════════════════════════════════════════════════════════════════════════════
# [교체] STEP 5-2 QMC 함수 (Cell 7 계속) — 상수 rule 허용, 원인 기록
# ════════════════════════════════════════════════════════════════════════════
def step5_2_learn_rules_qmc(binary_sorted, tf_tf_net, config):
    """
    t → t+1 transition 진리표를 Quine-McCluskey로 최소화.

    self-loop 발생 조건 (2가지, 명확히 구분):
      1. "no_inedge"      : tf_tf_net에 in-edge 없음 → rule = target_tf
      2. "constant_rule"  : on_set이 비어있어 rule이 항상 "0"
                            → 상수 "0" 허용 (self-loop 대체 안 함)
                            → SELFLOOP_REASON에 "constant_rule" 기록

    이전 버전과의 차이:
      - 상수 "0" → target_tf 대체 제거. 그대로 "0" 출력.
      - 상수 "1" → 그대로 "1" 출력.
      - 원인별 TF 목록을 최종 출력에서 구분하여 표시.
    """
    print("\n" + "="*70)
    print("STEP 5-2: Boolean Rule 학습 (Quine-McCluskey)")
    print("="*70)

    tf_cols     = [c for c in binary_sorted.columns
                   if c not in ("pseudotime", "pt_bin")]
    binary_data = binary_sorted[tf_cols].dropna().astype(int)

    # Sliding window
    window_size = config.get("window_size", 1)
    window_step = config.get("window_step", 1)
    X_list, Y_list = [], []

    if window_size <= 1:
        X_list = [binary_data.iloc[:-1].values]
        Y_list = [binary_data.iloc[1:].values]
    else:
        n = len(binary_data)
        for start in range(0, n - window_size, window_step):
            window  = binary_data.iloc[start: start + window_size]
            X_state = (window.mean() >= 0.5).astype(int).values
            ns      = start + window_step
            if ns >= n:
                break
            Y_state = binary_data.iloc[ns].values
            X_list.append(X_state.reshape(1, -1))
            Y_list.append(Y_state.reshape(1, -1))

    X_all = np.vstack(X_list)
    Y_all = np.vstack(Y_list)
    print(f"  전이 쌍 수: {len(X_all)}  "
          f"(window_size={window_size}, window_step={window_step})")

    rules_dict     = {}
    accuracy_dict  = {}
    selfloop_tfs   = []    # no_inedge
    constant0_tfs  = []    # constant_rule (on_set 비어있음)
    dontcare_stats = []

    SELFLOOP_REASON.clear()

    for i, target_tf in enumerate(tf_cols):
        regulators, is_selfloop = _select_regulators(
            target_tf, tf_cols, tf_tf_net, config
        )

        # 조건 1: in-edge 없음 → pure self-loop
        if is_selfloop:
            rules_dict[target_tf]    = target_tf
            accuracy_dict[target_tf] = None
            selfloop_tfs.append(target_tf)
            continue

        y_idx       = tf_cols.index(target_tf)
        y           = Y_all[:, y_idx]
        reg_indices = [tf_cols.index(r) for r in regulators]
        X           = X_all[:, reg_indices]
        n_vars      = len(regulators)

        # 진리표 구성
        truth_map = {}
        for x_row, y_val in zip(X, y):
            key = tuple(int(v) for v in x_row)
            truth_map.setdefault(key, {0: 0, 1: 0})
            truth_map[key][int(y_val)] += 1

        on_set, off_set, dc_set = [], [], []
        for minterm_idx in range(2 ** n_vars):
            bits = tuple(int(b) for b in format(minterm_idx, f"0{n_vars}b"))
            if bits in truth_map:
                counts = truth_map[bits]
                (on_set if counts[1] >= counts[0] else off_set).append(minterm_idx)
            else:
                dc_set.append(minterm_idx)

        dontcare_stats.append({
            "TF": target_tf, "n_vars": n_vars,
            "n_on": len(on_set), "n_off": len(off_set), "n_dc": len(dc_set),
            "coverage": round(len(truth_map) / (2 ** n_vars), 3),
        })

        # 조건 2: on_set 비어있음 → 상수 "0" 그대로 허용
        if not on_set:
            rules_dict[target_tf]    = "0"
            accuracy_dict[target_tf] = round((y == 0).mean(), 4)
            SELFLOOP_REASON[target_tf] = "constant_rule"
            constant0_tfs.append(target_tf)
            continue

        # on_set이 전체 (항상 1)
        if len(on_set) + len(dc_set) == 2 ** n_vars:
            rules_dict[target_tf]    = "1"
            accuracy_dict[target_tf] = round((y == 1).mean(), 4)
            continue

        # QMC 최소화
        prime_implicants = _qmc_find_prime_implicants(on_set, dc_set, n_vars)
        essential        = _qmc_select_essential(prime_implicants, on_set, n_vars)
        rule_str         = _qmc_to_bnet_expr(essential, regulators, n_vars)

        n_correct = sum(
            1 for x_row, y_val in zip(X, y)
            if _evaluate_bnet_expr(rule_str, regulators, x_row) == int(y_val)
        )
        acc = n_correct / len(y) if len(y) else 0.0

        rules_dict[target_tf]    = rule_str
        accuracy_dict[target_tf] = round(acc, 4)

        if i % 10 == 0:
            print(f"  [{i+1:>4}/{len(tf_cols)}] {target_tf:<20} "
                  f"acc={acc:.3f}  on={len(on_set)}  dc={len(dc_set)}  "
                  f"regs={regulators}")

    pd.DataFrame(dontcare_stats).to_csv(f"{OUT}/qmc_dontcare_stats.csv", index=False)
    accs = [v for v in accuracy_dict.values() if v is not None]

    print(f"\n  평균 accuracy  : {np.mean(accs):.4f}")
    print(f"  중앙값 accuracy: {np.median(accs):.4f}")

    # ── self-loop 원인 구분 출력 ──────────────────────────────────────────────
    print(f"\n  ── Self-loop / 상수 rule 요약 ──")
    print(f"  [원인 1] in-edge 없음      : {len(selfloop_tfs)} TFs")
    if selfloop_tfs:
        print(f"           → {selfloop_tfs}")
    print(f"  [원인 2] 상수 rule (always 0): {len(constant0_tfs)} TFs")
    if constant0_tfs:
        print(f"           → {constant0_tfs}")
    print(f"  ※ 원인 2 TF들은 binarization 후에도 거의 꺼진 상태입니다.")
    print(f"     auc_celltype_min_mean 또는 rss_threshold를 높이면 줄어듭니다.")

    return rules_dict, accuracy_dict


# ── QMC 내부 구현 ─────────────────────────────────────────────────────────

def _qmc_find_prime_implicants(on_set, dc_set, n_vars):
    """
    Quine-McCluskey: prime implicant 목록 생성.

    각 implicant 는 (mask, value) 튜플로 표현:
      mask  : 1 인 비트 = don't care 위치
      value : mask=0 인 비트의 실제 값
    예) n_vars=3, mask=0b010, value=0b001 → _1_ 패턴 (비트1 무관, 비트2=0, 비트0=1)
    """
    combined = set(on_set) | set(dc_set)

    # 초기화: 각 minterm → (mask=0, value=minterm)
    current = {(0, m) for m in combined}
    prime_implicants = set()

    while current:
        next_level = set()
        used       = set()

        items = list(current)
        for j in range(len(items)):
            for k in range(j + 1, len(items)):
                m1, v1 = items[j]
                m2, v2 = items[k]
                if m1 != m2:
                    continue
                diff = v1 ^ v2
                # diff 가 2의 거듭제곱이면 인접 → 합칠 수 있음
                if diff and (diff & (diff - 1)) == 0:
                    new_mask  = m1 | diff
                    new_value = v1 & v2
                    next_level.add((new_mask, new_value))
                    used.add(items[j])
                    used.add(items[k])

        # 합쳐지지 않은 implicant → prime implicant
        for imp in current:
            if imp not in used:
                prime_implicants.add(imp)

        current = next_level

    return list(prime_implicants)


def _qmc_select_essential(prime_implicants, on_set, n_vars):
    """
    Petrick's method 간소화 버전:
    on_set 의 모든 minterm 을 cover 하는 최소 prime implicant 집합 선택.
    """
    # 각 minterm 을 cover 하는 prime implicant 찾기
    coverage = {m: [] for m in on_set}
    for idx, (mask, value) in enumerate(prime_implicants):
        for m in on_set:
            if (m & ~mask) == (value & ~mask):
                coverage[m].append(idx)

    selected   = set()
    uncovered  = set(on_set)

    # 1차: essential prime implicant (딱 하나만 cover 하는 경우)
    for m, covers in coverage.items():
        if len(covers) == 1:
            selected.add(covers[0])

    # essential 로 cover 된 minterm 제거
    for idx in selected:
        mask, value = prime_implicants[idx]
        for m in list(uncovered):
            if (m & ~mask) == (value & ~mask):
                uncovered.discard(m)

    # 2차: greedy cover 나머지
    while uncovered:
        best_idx  = max(
            range(len(prime_implicants)),
            key=lambda idx: sum(
                1 for m in uncovered
                if (m & ~prime_implicants[idx][0]) ==
                   (prime_implicants[idx][1] & ~prime_implicants[idx][0])
            )
        )
        selected.add(best_idx)
        mask, value = prime_implicants[best_idx]
        for m in list(uncovered):
            if (m & ~mask) == (value & ~mask):
                uncovered.discard(m)

    return [prime_implicants[i] for i in selected]


def _qmc_to_bnet_expr(essential_pis, var_names, n_vars):
    """선택된 prime implicant → .bnet SOP expression 변환."""
    if not essential_pis:
        return "0"
    products = []
    for mask, value in essential_pis:
        literals = []
        for bit in range(n_vars - 1, -1, -1):
            var_idx = (n_vars - 1) - bit
            if var_idx >= len(var_names):
                continue
            if (mask >> bit) & 1:
                continue
            vname = var_names[var_idx]
            literals.append(vname if (value >> bit) & 1 else f"!{vname}")
        if not literals:
            products.append("1")
        elif len(literals) == 1:
            products.append(literals[0])
        else:
            products.append("(" + " & ".join(literals) + ")")
    if len(products) == 1:
        return products[0]
    return "(" + " | ".join(products) + ")"


def _evaluate_bnet_expr(expr, var_names, values):
    """
    .bnet expression 을 주어진 binary 값으로 평가 (accuracy 계산용).
    expr  : bnet 문자열
    values: var_names 순서에 맞는 0/1 배열
    """
    env = {name: int(val) for name, val in zip(var_names, values)}
    # bnet → Python 문법 변환
    py_expr = expr.replace("!", " not ").replace("&", " and ").replace("|", " or ")
    try:
        result = eval(py_expr, {"__builtins__": {}}, env)
        return int(bool(result))
    except Exception:
        return 0


# ════════════════════════════════════════════════════════════════════════════
# STEP 5 진입점 — rule_method 로 5-1 / 5-2 선택
# ════════════════════════════════════════════════════════════════════════════
def step5_learn_boolean_rules(binary_sorted, tf_tf_net, config):
    """
    CONFIG["rule_method"] 에 따라 학습 방식 선택:
      "decision_tree"  (기본) → step5_1  Decision Tree / REVEAL
      "qmc"                   → step5_2  Quine-McCluskey
    """
    method = config.get("rule_method", "decision_tree")
    print(f"\n  Rule learning method: '{method}'")

    if method == "qmc":
        return step5_2_learn_rules_qmc(binary_sorted, tf_tf_net, config)
    else:
        return step5_1_learn_rules_decision_tree(binary_sorted, tf_tf_net, config)

In [681]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 6: .bnet 파일 저장 + SBML 변환 (선택)
# ════════════════════════════════════════════════════════════════════════════
def step6_export_network(rules_dict, tf_tf_net, accuracy_dict, config):
    """
    .bnet (PyBoolNet 표준) 및 edge list 저장.
    선택적으로 SBML-qual 형식으로 변환.
    """
    print("\n" + "="*70)
    print("STEP 6: 네트워크 파일 저장")
    print("="*70)

    bnet_path = f"{OUT}/pdac_boolean_network.bnet"

    with open(bnet_path, "w") as f:
        f.write("targets, factors\n")
        for tf, rule in sorted(rules_dict.items()):
            acc = accuracy_dict.get(tf)
            acc_str = f"acc={acc:.3f}" if acc is not None else "self"
            f.write(f"{tf},\t{rule}\t# {acc_str}\n")

    print(f"  .bnet 저장: {bnet_path}")
    print(f"  총 {len(rules_dict)} TF 규칙")

    # ── 요약 통계 저장 ───────────────────────────────────────────────────────
    summary = []
    for tf, rule in rules_dict.items():
        n_regulators = len(tf_tf_net[tf_tf_net["target"] == tf])
        summary.append({
            "TF"          : tf,
            "rule"        : rule,
            "n_regulators": n_regulators,
            "accuracy"    : accuracy_dict.get(tf)
        })
    pd.DataFrame(summary).to_csv(f"{OUT}/network_rules_summary.csv", index=False)

    # ── SBML-qual 변환 (libsbml 설치된 경우) ─────────────────────────────────
    try:
        _export_sbml(rules_dict, tf_tf_net)
    except ImportError:
        print("  SBML 변환 스킵 (libsbml 미설치). pip install python-libsbml 후 재실행")

    return bnet_path


def _export_sbml(rules_dict, tf_tf_net):
    """SBML-qual 형식으로 Boolean network 내보내기 (libsbml 필요)."""
    import libsbml

    writer   = libsbml.SBMLWriter()
    document = libsbml.SBMLDocument(3, 1)
    document.enablePackage(
        libsbml.QualExtension.getXmlnsL3V1V1(), "qual", True
    )
    model = document.createModel()
    model.setId("PDAC_Boolean_Network")

    qual_plugin = model.getPlugin("qual")

    tfs = sorted(rules_dict.keys())
    for tf in tfs:
        qs = qual_plugin.createQualitativeSpecies()
        qs.setId(tf.replace("-", "_"))
        qs.setMaxLevel(1)
        qs.setInitialLevel(0)
        qs.setCompartment("default")

    for tf, rule in rules_dict.items():
        trans = qual_plugin.createTransition()
        trans.setId(f"tr_{tf.replace('-','_')}")
        output = trans.createOutput()
        output.setQualitativeSpecies(tf.replace("-", "_"))
        output.setTransitionEffect(
            libsbml.OUTPUT_TRANSITION_EFFECT_ASSIGNMENT_LEVEL
        )
        ft = trans.createFunctionTerm()
        ft.setResultLevel(1)
        # rule을 MathML로 변환 (간단한 경우만)
        # 복잡한 경우는 libsbml ASTNode 빌더 필요

    sbml_path = f"{OUT}/pdac_boolean_network.sbml"
    writer.writeSBMLToFile(document, sbml_path)
    print(f"  SBML 저장: {sbml_path}")


In [682]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 6-2: 노드 제거 (자동 + 수동)
# ════════════════════════════════════════════════════════════════════════════
def step6_2_remove_nodes(rules_dict, tf_tf_net, accuracy_dict,
                          auto_remove=True, manual_remove=None):
    """
    [자동 제거] auto_remove=True 일 때:
      조건 1: rule이 상수("0", "1") 또는 self-loop (rule == TF 자신)
      조건 2: Step 5에서 학습된 다른 TF의 rule 문자열에
              input literal로 등장하지 않음
              (tf_tf_net 구조가 아닌 실제 학습된 logic 기준)
      → 두 조건 모두 만족하는 노드 제거

    [수동 제거] manual_remove 리스트에 TF명을 직접 입력:
      → 조건 무관하게 무조건 제거
      → 자동 제거 후 남은 노드에 추가 적용
    """
    import re

    print("\n" + "="*70)
    print("STEP 6-2: 노드 제거")
    print("="*70)
    print(f"  시작: {len(rules_dict)} nodes, {len(tf_tf_net)} edges")

    removed_auto   = []
    removed_manual = []

    # ── 자동 제거 ────────────────────────────────────────────────────────────
    if auto_remove:
        # 조건 1: 상수 또는 self-loop rule 노드 후보
        constant_tfs = {
            tf for tf, rule in rules_dict.items()
            if rule.strip() in {"0", "1", tf}
        }

        # 조건 2: Step 5 rule 문자열을 직접 파싱
        # 각 TF가 다른 TF의 rule에 input literal로 등장하는지 확인
        #
        # bnet rule 예시: "(FOS & !FOSB) | (ATF3 & KLF4)"
        # → FOS, FOSB, ATF3, KLF4 가 input으로 등장
        # → "!FOSB" 에서 FOSB를 추출 (! 제거)
        # → 부분문자열 오매칭 방지: word boundary 기준 토큰 파싱
        appears_as_input = set()

        for target_tf, rule in rules_dict.items():
            rule_stripped = rule.strip()
            # 상수/self-loop rule은 input이 없으므로 파싱 불필요
            if rule_stripped in {"0", "1"} or rule_stripped == target_tf:
                continue
            # rule에서 TF 이름 토큰 추출 (앞의 ! 제거)
            # bnet 토큰: 알파벳으로 시작하는 알파뉴메릭 + 하이픈/언더스코어
            tokens = re.findall(r"!?([A-Za-z][A-Za-z0-9_\-]*)", rule_stripped)
            for tok in tokens:
                # 자기 자신은 제외, rules_dict에 있는 TF만
                if tok != target_tf and tok in rules_dict:
                    appears_as_input.add(tok)

        # 제거 대상: 상수/self-loop이면서 어떤 rule에도 input으로 안 쓰임
        removed_auto = sorted(constant_tfs - appears_as_input)

        if removed_auto:
            removed_c0 = [tf for tf in removed_auto
                          if rules_dict[tf].strip() == "0"]
            removed_c1 = [tf for tf in removed_auto
                          if rules_dict[tf].strip() == "1"]
            removed_sl = [tf for tf in removed_auto
                          if rules_dict[tf].strip() == tf]
            print(f"\n  [자동] 고립 상수 노드: {len(removed_auto)}개 제거")
            print(f"    상수 0    : {len(removed_c0)}개  {removed_c0}")
            print(f"    상수 1    : {len(removed_c1)}개  {removed_c1}")
            print(f"    self-loop : {len(removed_sl)}개  {removed_sl}")

            # 참고: tf_tf_net source 기준과 비교
            source_tfs    = set(tf_tf_net["source"].tolist())
            net_based     = sorted(constant_tfs - source_tfs)
            logic_only    = sorted(set(removed_auto) - set(net_based))
            net_only      = sorted(set(net_based) - set(removed_auto))
            
        else:
            print("\n  [자동] 제거 대상 없음")

    # ── 수동 제거 ────────────────────────────────────────────────────────────
    if manual_remove:
        remaining      = set(rules_dict.keys()) - set(removed_auto)
        removed_manual = sorted(set(manual_remove) & remaining)
        not_found      = sorted(set(manual_remove) - set(rules_dict.keys()))

        print(f"\n  [수동] 지정 노드: {len(removed_manual)}개 제거")
        print(f"    제거: {removed_manual}")
        if not_found:
            print(f"    ⚠ 네트워크에 없어 스킵: {not_found}")

    # ── 통합 제거 실행 ────────────────────────────────────────────────────────
    all_removed = set(removed_auto) | set(removed_manual)

    rules_dict_clean    = {tf: r for tf, r in rules_dict.items()
                           if tf not in all_removed}
    accuracy_dict_clean = {tf: a for tf, a in accuracy_dict.items()
                           if tf not in all_removed}
    tf_tf_net_clean     = tf_tf_net[
        ~tf_tf_net["source"].isin(all_removed) &
        ~tf_tf_net["target"].isin(all_removed)
    ].reset_index(drop=True)

    print(f"\n  결과: {len(rules_dict_clean)} nodes, "
          f"{len(tf_tf_net_clean)} edges  (총 {len(all_removed)}개 제거)")

    # 제거 목록 저장
    removed_records = []
    for tf in sorted(all_removed):
        rule = rules_dict.get(tf, "N/A")
        removed_records.append({
            "TF"          : tf,
            "rule"        : rule,
            "removal_type": "auto" if tf in removed_auto else "manual",
            "reason"      : (
                "constant_0"    if rule.strip() == "0"  else
                "constant_1"    if rule.strip() == "1"  else
                "selfloop_sink" if rule.strip() == tf   else
                "manual"
            ),
            "accuracy"    : accuracy_dict.get(tf),
        })
    pd.DataFrame(removed_records).to_csv(
        f"{OUT}/step6_2_removed_nodes.csv", index=False
    )
    print(f"  → {OUT}/step6_2_removed_nodes.csv 저장")

    return rules_dict_clean, tf_tf_net_clean, accuracy_dict_clean

In [683]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 7: Attractor → Cell 매핑 + UMAP 시각화 (Euclidean distance)
# ════════════════════════════════════════════════════════════════════════════
def step7_attractor_cell_mapping(attractor_df_path, cell_binary, adata, config,
                                  dist_threshold=None):
    """
    각 attractor state에서 가장 가까운 cell을 Euclidean distance로 찾아
    UMAP 위에 attractor 위치를 원형으로 표시.
 
    Cyclic attractor 처리:
      BoolNet cyclic attractor는 0/1 사이 소수값 (0.5, 0.33 등)으로 출력됨.
      → 이진화 없이 소수값 그대로 Euclidean distance 계산.
      → cyclic TF가 0.5이면 cell 0/1 여부와 무관하게 거리 0.5로 동등 취급
         (해당 TF는 attractor 매핑에 중립적으로 기여).
      → cyclic TF가 0.33이면 cell=0에 더 가깝게 취급.
 
    Euclidean vs Hamming (이전 버전):
      - Steady state TF (0 or 1): 두 방식 결과 동일
      - Cyclic TF (소수값): Euclidean만 소수값 반영 가능
      - Euclidean은 부분적으로 일치하는 정도를 거리에 연속적으로 반영
 
    원형 크기:
      attractor의 size 컬럼 값에 선형 비례.
      (size 최솟값 → 면적 100pt, 최댓값 → 면적 1000pt)
 
    Parameters
    ----------
    attractor_df_path : str
        R BoolNet에서 저장한 attractor CSV 경로.
        형식: rows=attractors, columns=TF nodes + 마지막 "size" column
    cell_binary : pd.DataFrame
        step4_2 결과. index=cell_id, columns=TF + "cell_type" + "pseudotime"
    adata : AnnData
        obsm["X_umap"] 포함
    config : dict
    dist_threshold : float or None
        None=모든 attractor 매핑.
        float=가장 가까운 cell까지의 Euclidean distance가 이 값 초과이면
        UMAP에 "매핑 불가"로 표시 (CSV에는 기록됨).
 
    출력 파일
    ----------
    attractor_cell_mapping.csv    : attractor별 매핑 결과
    attractor_umap.pdf/png        : UMAP 시각화 (원형 크기 = attractor size)
    attractor_euclidean_dist.pdf  : attractor별 Euclidean distance 막대 그래프
    """
    print("\n" + "="*70)
    print("STEP 7: Attractor → Cell 매핑 + UMAP 시각화 (Euclidean distance)")
    print("="*70)
 
    # ── 7-1. Attractor DataFrame 로드 ────────────────────────────────────────
    attr_df  = pd.read_csv(attractor_df_path)
    size_col = [c for c in attr_df.columns if c == "Attr_Size"]
    attr_size = attr_df[size_col[0]].values.astype(float) if size_col else None
    node_cols = [c for c in attr_df.columns if c != "Attr_Size"]
 
    # 소수값 그대로 유지 (이진화 없음)
    attr_raw  = attr_df[node_cols].astype(float)
    is_cyclic = ~attr_raw.isin([0.0, 1.0]).all(axis=1)
 
    print(f"  Attractor 수   : {len(attr_df)}")
    print(f"  Attractor nodes: {len(node_cols)}")
    if is_cyclic.any():
        print(f"\n  Cyclic attractor: {is_cyclic.sum()}개  "
              f"(소수값 그대로 Euclidean 계산, 이진화 없음)")
        for idx in attr_raw.index[is_cyclic]:
            non_binary = attr_raw.loc[idx][
                ~attr_raw.loc[idx].isin([0.0, 1.0])
            ]
            print(f"    Attr {idx}: {non_binary.to_dict()}")
    if attr_size is not None:
        print(f"  Attractor sizes: {attr_size.tolist()}")
 
    # ── 7-2. 공통 TF 추출 ────────────────────────────────────────────────────
    meta_cols  = {"cell_type", "pseudotime"}
    tf_in_cell = [c for c in cell_binary.columns if c not in meta_cols]
    common_tfs = [tf for tf in node_cols if tf in tf_in_cell]
    missing    = [tf for tf in node_cols if tf not in tf_in_cell]
 
    print(f"\n  공통 TF: {len(common_tfs)} / {len(node_cols)}")
    if missing:
        print(f"  ⚠ cell_binary에 없는 TF (제외): {missing}")
 
    # cell: float 0/1, attractor: float (소수 포함)
    cell_mat = cell_binary[common_tfs].values.astype(float)  # cell × TF
    attr_mat = attr_raw[common_tfs].values.astype(float)     # attr × TF
    n_cells  = cell_mat.shape[0]
    n_attrs  = attr_mat.shape[0]
    n_tfs    = len(common_tfs)
 
    # ── 7-3. Euclidean distance (attractor × cell) ───────────────────────────
    print(f"\n  Euclidean distance 계산: {n_attrs} attractors × {n_cells} cells ...")
 
    # Broadcasting: (n_attrs, 1, n_tfs) vs (1, n_cells, n_tfs)
    attr_expanded = attr_mat[:, np.newaxis, :]    # (n_attrs, 1,       n_tfs)
    cell_expanded = cell_mat[np.newaxis, :, :]    # (1,       n_cells, n_tfs)
    diff          = attr_expanded - cell_expanded  # (n_attrs, n_cells, n_tfs)
    euclid_mat    = np.sqrt((diff ** 2).sum(axis=2))  # (n_attrs, n_cells)
 
    # 정규화: 최대 가능 거리 = sqrt(n_tfs) (모든 TF가 0 vs 1일 때)
    max_dist   = np.sqrt(n_tfs)
    euclid_norm = euclid_mat / max_dist
 
    best_cell_idx  = euclid_mat.argmin(axis=1)       # (n_attrs,)
    best_dist      = euclid_mat.min(axis=1)          # (n_attrs,)
    best_dist_norm = euclid_norm.min(axis=1)         # (n_attrs,)
    best_cell_ids  = [cell_binary.index[i] for i in best_cell_idx]
 
    # ── 7-4. 매핑 결과 DataFrame ─────────────────────────────────────────────
    attr_names = [str(idx) for idx in attr_df.index]
    records    = []
    for ai, attr_name in enumerate(attr_names):
        rec = {
            "attractor"          : attr_name,
            "is_cyclic"          : bool(is_cyclic.iloc[ai]),
            "mapped_cell_id"     : best_cell_ids[ai],
            "euclidean_dist"     : round(float(best_dist[ai]), 4),
            "euclidean_dist_norm": round(float(best_dist_norm[ai]), 4),
            "mappable"           : (dist_threshold is None or
                                    best_dist[ai] <= dist_threshold),
        }
        if attr_size is not None:
            rec["size"] = int(attr_size[ai])
        if "cell_type" in cell_binary.columns:
            rec["mapped_cell_type"] = cell_binary.loc[
                best_cell_ids[ai], "cell_type"]
        if "pseudotime" in cell_binary.columns:
            rec["mapped_pseudotime"] = round(float(
                cell_binary.loc[best_cell_ids[ai], "pseudotime"]), 4)
        records.append(rec)
 
    result_df = pd.DataFrame(records)
    result_df.to_csv(f"{OUT}/attractor_cell_mapping.csv", index=False)
 
    print(f"\n  매핑 결과:")
    for _, row in result_df.iterrows():
        cyc_str  = " [cyclic]" if row["is_cyclic"] else ""
        map_warn = "  ⚠ threshold 초과" if not row["mappable"] else ""
        ct_str   = (f"  [{row['mapped_cell_type']}]"
                    if "mapped_cell_type" in row else "")
        print(f"    Attr {row['attractor']:>4}{cyc_str}  →  "
              f"cell {row['mapped_cell_id']}"
              f"  (euclid={row['euclidean_dist']:.3f}"
              f", norm={row['euclidean_dist_norm']:.3f})"
              f"{ct_str}{map_warn}")
    print(f"  → {OUT}/attractor_cell_mapping.csv 저장")
# ── 7-5. UMAP 시각화 ─────────────────────────────────────────────────────
    if "X_umap" not in adata.obsm:
        print("  ⚠ adata.obsm['X_umap'] 없음 — UMAP 시각화 스킵")
        return result_df

    cell_id_to_adata_idx = {c: i for i, c in enumerate(adata.obs_names)}
    umap_all = adata.obsm["X_umap"]

    # 1. Cell Type 컬러맵 설정 (adata.uns에서 가져오기)
    ct_color_map = {}
    if "cell_type" in cell_binary.columns:
        if "cell_type" in adata.obs.columns and "cell_type_colors" in adata.uns:
            # adata.uns 정보를 사용하여 매핑
            categories = adata.obs["cell_type"].cat.categories
            colors = adata.uns["cell_type_colors"]
            ct_color_map = dict(zip(categories, colors))
        else:
            # 정보가 없으면 기본 Set2 사용
            unique_cts = cell_binary["cell_type"].unique()
            ct_palette = plt.cm.get_cmap("Set2", len(unique_cts))
            ct_color_map = {ct: ct_palette(i) for i, ct in enumerate(unique_cts)}

    # 2. 원 크기: 전체 대비 비율(%) 계산
    if attr_size is not None:
        size_vals = result_df["size"].values.astype(float)
        total_size = size_vals.sum()
        rel_sizes = size_vals / total_size
        marker_sizes = 200 + (rel_sizes * 10000) 
    else:
        marker_sizes = np.full(n_attrs, 1000.0)

    fig, ax = plt.subplots(figsize=(10, 8))

    # 배경: 전체 cell 그리기
    common_in_adata = [c for c in cell_binary.index if c in cell_id_to_adata_idx]
    bg_idx = [cell_id_to_adata_idx[c] for c in common_in_adata]
    bg_coords = umap_all[bg_idx]

    legend_handles = []
    if "cell_type" in cell_binary.columns:
        # ct_color_map에 있는 순서대로 그리기 (범례 순서 보장)
        for ct, color in ct_color_map.items():
            # cell_binary 데이터 기준으로 마스크 생성
            ct_mask = np.array([cell_binary.loc[c, "cell_type"] == ct for c in common_in_adata])
            if ct_mask.sum() > 0:
                s = ax.scatter(bg_coords[ct_mask, 0], bg_coords[ct_mask, 1],
                               c=[color], s=5, alpha=0.25, label=ct, rasterized=True)
                legend_handles.append(s)
        
        # 첫 번째 범례 (Cell Type) 추가
        leg1 = ax.legend(handles=legend_handles, loc='upper right', title="Cell Type", fontsize=11)
        ax.add_artist(leg1)
    else:
        ax.scatter(bg_coords[:, 0], bg_coords[:, 1], c="#cccccc", s=5, alpha=0.3, rasterized=True)

    # 전경: attractor 원형
    for ai, (_, row) in enumerate(result_df.iterrows()):
        if not row["mappable"]: continue
        cell_id = row["mapped_cell_id"]
        if cell_id not in cell_id_to_adata_idx: continue

        uidx = cell_id_to_adata_idx[cell_id]
        x, y = umap_all[uidx, 0], umap_all[uidx, 1]
        
        mapped_ct = row.get("mapped_cell_type", None)
        edge_color = ct_color_map.get(mapped_ct, "black") 
        
        ax.scatter(x, y, s=marker_sizes[ai], facecolors="#d3d3d3", edgecolors=edge_color,
                   linewidths=2.0, alpha=0.7, zorder=5)

    # 3. 커스텀 사이즈 범례 (0.1%, 1%, 10%)
    from matplotlib.lines import Line2D
    def get_s(pct): return 200 + (pct/100 * 10000)
    
    size_legend = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#d3d3d3', 
               markeredgecolor='black', markersize=np.sqrt(get_s(0.1))/1, label='0.1%'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#d3d3d3', 
               markeredgecolor='black', markersize=np.sqrt(get_s(1.0))/1, label='1%'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#d3d3d3', 
               markeredgecolor='black', markersize=np.sqrt(get_s(10.0))/1, label='10%')
    ]
    ax.legend(
        handles=size_legend, 
        loc='lower left', 
        title="Basin Size (%)", 
        fontsize=8,
        borderpad=3,      # 범례 상자 내부 여백
        labelspacing=3,   # [중요] 항목(0.1%, 1%, 10%) 간의 세로 간격(겹침 방지)
        handletextpad=1.5,  # 마커와 글자 사이의 가로 간격
        borderaxespad=1.0   # 상자와 그래프 모서리 사이의 간격
    )
    ax.set_title("Attractor Positions on UMAP", fontsize=12, fontweight="bold")
    ax.set_xlabel("UMAP1", fontsize=10)
    ax.set_ylabel("UMAP2", fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f"{OUT}/attractor_umap.pdf", dpi=150, bbox_inches="tight")
    plt.savefig(f"{OUT}/attractor_umap.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  → attractor_umap.pdf/png 저장")
    
    # 막대 그래프를 위한 컬러맵 정의
    palette = plt.cm.get_cmap("tab10", n_attrs) 
    attr_colors = {name: palette(i) for i, name in enumerate(attr_names)}
 
    # ── 7-6. Euclidean distance 막대 그래프 ──────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(5, n_attrs * 1.2), 4))
    bar_labels = [
        f"Attr {r['attractor']}" + (" ↻" if r["is_cyclic"] else "")
        for _, r in result_df.iterrows()
    ]
    bars = ax.bar(
        bar_labels,
        result_df["euclidean_dist"].values,
        color=[attr_colors[str(r["attractor"])]
               for _, r in result_df.iterrows()],
        edgecolor="black", linewidth=0.8,
    )
    ax.bar_label(
        bars,
        labels=[f"{v:.3f}" for v in result_df["euclidean_dist"].values],
        padding=3, fontsize=9
    )
    ax.axhline(max_dist, color="#aaa", linestyle=":",
               linewidth=1,
               label=f"max possible = √{n_tfs} = {max_dist:.2f}")
    if dist_threshold is not None:
        ax.axhline(dist_threshold, color="red", linestyle="--",
                   linewidth=1.5, label=f"threshold={dist_threshold}")
    ax.set_ylabel("Euclidean distance", fontsize=10)
    ax.set_title("Distance from Each Attractor to Nearest Cell",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{OUT}/attractor_euclidean_dist.pdf", dpi=150, bbox_inches="tight")
    plt.savefig(f"{OUT}/attractor_euclidean_dist.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → attractor_euclidean_dist.pdf/png 저장")
# ── 7-7. Cell Type별 Attractor 요약 통계 계산 ──────────────────────────────
    # 매핑 가능한(mappable) attractor만 대상으로 계산
    mappable_df = result_df[result_df["mappable"] == True].copy()
    
    if "mapped_cell_type" in mappable_df.columns and "size" in mappable_df.columns:
        # 그룹별 개수 및 사이즈 합산
        summary = mappable_df.groupby("mapped_cell_type").agg(
            attractor_count=("attractor", "count"),
            total_basin_size=("size", "sum")
        ).sort_values("total_basin_size", ascending=False)
        
        # 비율 추가 (선택사항)
        summary["size_percentage"] = (summary["total_basin_size"] / summary["total_basin_size"].sum()) * 100
        print(f"{OUT}/nominal_attractor_celltype.csv" + "에 nominal 정보 저장")
        summary.to_csv(f"{OUT}/nominal_attractor_celltype.csv")
        print("\n" + "="*60)
        print("Attractor Mapping Summary by Cell Type")
        print("="*60)
        print(summary)
        print("="*60) 
        
    return result_df

In [684]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 8: 네트워크 시각화
# ════════════════════════════════════════════════════════════════════════════
def step8_visualize_network(G, tf_tf_net, attr_df=None):
    """
    TF-TF Boolean network 시각화:
      - 노드 크기 = degree
      - 엣지 색 = activation(파랑) / inhibition(빨강)
      - PDAC attractor active TF 강조
    """
    print("\n" + "="*70)
    print("STEP 8: 네트워크 시각화")
    print("="*70)

    if G.number_of_nodes() == 0:
        print("  네트워크 비어있음, 시각화 스킵")
        return

    degree_dict = dict(G.degree())
    top_nodes   = sorted(degree_dict, key=degree_dict.get, reverse=True)[:]
    subG        = G.subgraph(top_nodes).copy()

    fig, ax = plt.subplots(figsize=(14, 12))
    pos = nx.spring_layout(subG, seed=42, k=1.5)

    # 노드 속성
    node_sizes  = [degree_dict.get(n, 1) * 80 + 200 for n in subG.nodes()]
    # PDAC attractor active TF 강조
    pdac_active = set()
    if attr_df is not None and len(attr_df) > 0:
        top_attr = attr_df.iloc[0].drop(["attractor_id", "PDAC_score"])
        pdac_active = set(top_attr[top_attr == 1].index)

    node_colors = [
        "#E74C3C" if n in pdac_active else "#3498DB"
        for n in subG.nodes()
    ]

    # 엣지 속성
    edge_colors = []
    for u, v, d in subG.edges(data=True):
        edge_colors.append("#2980B9" if d.get("sign", 1) == 1 else "#C0392B")

    nx.draw_networkx_nodes(subG, pos, node_size=node_sizes,
                           node_color=node_colors, alpha=0.85, ax=ax)
    nx.draw_networkx_labels(subG, pos, font_size=7, ax=ax)
    nx.draw_networkx_edges(subG, pos, edge_color=edge_colors,
                           arrows=True, arrowsize=15,
                           connectionstyle="arc3,rad=0.1",
                           width=1.5, alpha=0.7, ax=ax)

    # Legend
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Patch(facecolor="#3498DB", label="TF; (Size: Degree)"),
        Line2D([0],[0], color="#2980B9", linewidth=2, label="Activation"),
        Line2D([0],[0], color="#C0392B", linewidth=2, label="Inhibition"),
    ]
    ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
    ax.set_title("PDAC Transition TF Boolean Network",
                 fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{OUT}/tf_network_visualization.pdf", dpi=150)
    plt.savefig(f"{OUT}/tf_network_visualization.png", dpi=150)
    plt.close()
    print(f"  → 네트워크 시각화 저장: tf_network_visualization.pdf/png")


In [685]:
# ── Step 1: 전처리 ────────────────────────────────────────────────────────
adata, tf_in_data = step1_load_and_preprocess(CONFIG)


STEP 1: 데이터 로드 & 전처리 (HVG 계산 스킵)
  로드된 데이터: 15,562 cells × 3,000 genes
  [알림] 이미 HVG 필터링이 완료된 데이터를 사용합니다.
  Pseudotime CSV 로드: /data/yeohs0212/MM/GSE207938/pseudotime_results.csv
  Pseudotime 필터링 완료: 15,562 cells 남음
  Cell type 필터 ['ADM', 'Acinar', 'Bridge', 'PDAC']: 15,562 cells 남음
  raw count: adata.layers['counts'] → adata.X

  현재 데이터 유지: 3,000 genes (HVG 이미 적용됨)
  데이터 내 TF 확인: 215 TFs found
  adata.X         → log1p(CPM)  [GRNBoost2 입력]
  layers['raw_counts'] → integer count  [AUCell 입력]

  ✅ Step 1 완료: 15,562 cells × 3,000 genes (pseudotime 정렬 완료)


In [686]:
# ── Step 2: pySCENIC 네트워크 node 선정────────────────────────────────────────────────
ct_series = adata.obs[CONFIG["celltype_key"]]   # cell type 정보 주입
adjacencies, regulon_dict, auc_mtx = step2_pyscenic_grn(
    adata, tf_in_data, CONFIG, ct_series=ct_series
)
# 48 TFs


STEP 2: pySCENIC GRN 추론
  [표현량] GRNBoost2 입력: log1p-CPM  (15562, 3000)
  [표현량] AUCell 입력:    raw count   (15562, 3000)

  [2-1] GRNBoost2 실행 중... (10~60분 소요)
preparing dask client
parsing input
creating dask graph
14 partitions
computing dask graph
shutting down client and local cluster
finished
       → 217547 TF-gene links 저장

  [2-2] cisTarget regulon pruning 중... (DB 2개 병렬 사용)
         • hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
         • hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather



2026-05-07 11:42:36,446 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-05-07 11:42:36,871 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2026-05-07 11:42:42,446 - pyscenic.utils - INFO - Creating modules.

2026-05-07 11:42:59,524 - pyscenic.prune - INFO - Using 8 workers.

2026-05-07 11:42:59,524 - pyscenic.prune - INFO - Using 8 workers.

2026-05-07 11:43:02,591 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp


2026-05-07 11:43:17,269 - pyscenic.transform - WARNING - Less than 80% of the genes in CERS4 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:17,490 - pyscenic.transform - WARNING - Less than 80% of the genes in CREB3L4 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:17,553 - pyscenic.transform - WARNING - Less than 80% of the genes in CERS4 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:17,664 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for P4HB could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:17,779 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for PAX6 could be mapped to hg38_500bp_up_1


2026-05-07 11:43:21,899 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for CREB3L4 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:22,023 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for DAB2 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:22,379 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for DBP could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:22,570 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for SOX2 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:22,595 - pyscenic.transform - WARNING - Less than 80% of the genes in ATF6B could be map


2026-05-07 11:43:26,784 - pyscenic.transform - WARNING - Less than 80% of the genes in H1FX could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:26,970 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for UBTF could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:27,088 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for EP300 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:27,171 - pyscenic.transform - WARNING - Less than 80% of the genes in DPF1 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:43:27,211 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for EPAS1 could be mapped to hg38_5

...


2026-05-07 11:44:15,790 - pyscenic.transform - WARNING - Less than 80% of the genes in MYC could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:16,071 - pyscenic.transform - WARNING - Less than 80% of the genes in NCOR1 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:16,073 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for PLAGL1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:16,204 - pyscenic.transform - WARNING - Less than 80% of the genes in SETBP1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:16,255 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for POLR2A could be mapped to hg38_500bp_up_


2026-05-07 11:44:19,262 - pyscenic.transform - WARNING - Less than 80% of the genes in SP100 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:19,321 - pyscenic.transform - WARNING - Less than 80% of the genes in MECOM could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:19,393 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather(3): All regulons derived.

2026-05-07 11:44:19,393 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather(3): All regulons derived.

2026-05-07 11:44:19,402 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather(3): Done.

2026-05-07 11:44:19,402 - pyscenic.prune - INFO - Worker hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_mo


2026-05-07 11:44:24,520 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for STAT1 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:24,621 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for IRF7 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:24,699 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for TAF7 could be mapped to hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:24,741 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for UBTF could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:24,779 - pyscenic.transform - WARNING - Less than 80% of the genes in P4HB could be mappe


2026-05-07 11:44:32,597 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather(1): Done.

2026-05-07 11:44:32,597 - pyscenic.prune - INFO - Worker hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather(1): Done.

2026-05-07 11:44:35,508 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for MAFB could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:38,098 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for MEF2C could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:38,409 - pyscenic.transform - WARNING - Less than 80% of the genes in Regulon for MXD1 could be mapped to hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather. Skipping this module.

2026-05-07 11:44:40,877 - pyscenic.prune 

Create regulons from a dataframe of enriched features.
Additional columns saved: []
       → 61 regulons 추론됨

  [2-3] AUCell TF activity 계산 중...
       → AUC matrix (필터 전): (15562, 61)

  [2-4] RSS 기반 TF 필터
         RSS matrix: (4, 61)  → ./PDAC_boolean_output/rss_scores.csv
         rss_threshold: 0.3
         TF: 61 → 48 유지 (13 제거)

         Cell type별 top-5 RSS TF:
           Acinar              : ['SOX18', 'ELK3', 'MSX1', 'RBPJL', 'PTF1A']
           ADM                 : ['PTF1A', 'RBPJL', 'FOSB', 'ATF3', 'HES1']
           Bridge              : ['HMGA1', 'HMGA2', 'GATA5', 'YBX1', 'SOX11']
           PDAC                : ['HMGA2', 'KLF10', 'HMGA1', 'CEBPB', 'MAFF']
         RSS 히트맵 저장: ./PDAC_boolean_output/rss_heatmap.pdf/png

       → 최종 AUC matrix: (15562, 48)


In [687]:
# ── Step 3: TF-TF 네트워크 추출────────────────────────────────────────────────
tf_tf_net, G = step3_extract_tf_tf_network(
    regulon_dict, auc_mtx, tf_in_data, CONFIG
)



STEP 3: TF-TF 네트워크 추출  (regulon 내 AUC 반상관 기반)
  inhibition_r_threshold : r_auc < -0.00  → inhibition
  min_edge_corr          : |r_auc| ≥ 0.05
  엣지 후보 범위         : regulon에 포함된 TF→TF pair만

  regulon 내 TF→TF pair: 236개 (필터 전)
  |r_auc| ≥ 0.05 필터 후: 229개

  r_auc 분포 (|r| ≥ 0.05 기준):
    r < 0  : 13  → inhibition 후보
    r > 0  : 216  → activation 후보

  최종 TF-TF edges : 229
    Activation (+)  : 216  (94.3%)
    Inhibition (−)  : 13  (5.7%)

  → r 분포 히스토그램 저장: tf_tf_r_distribution.pdf/png
  Graph: 42 nodes, 229 edges


In [688]:
# ── Step 3-2: 필터링 된 TF-TF 네트워크에 존재하는 노드로 node list 추출 ──────────────────────────────────────────

node_list = list(set(tf_tf_net["source"].tolist() + tf_tf_net["target"].tolist()))

len(node_list) #42

42

In [689]:
# ── Step 4: Binarization ─────────────────────────────────────────────────
binary_sorted, tf_cols, gmm_thresholds = step4_binarize_tf_activity(
    auc_mtx[node_list], adata, CONFIG
)


STEP 4: TF Activity Binarization (pseudotime 기반)
  Binarized TFs: 42 (원본 42개 에서 min_cells_acitve에 의해 필터링 됨)
  → 시각화 저장: tf_activity_heatmap.pdf/png


In [690]:
 # ── Step 4-2: Cell별 / Cell state별 binary state ─────────────────────────
cell_binary, cellstate_profile = step4_2_binary_state_profiles(
    auc_mtx[node_list], adata, tf_tf_net, gmm_thresholds, CONFIG
)
 


STEP 4-2: Cell state별 / Cell별 binary TF state 계산
  Boolean network TF 수: 42
  Cell별 binary matrix → cell_binary_state.csv
  Cell state 수: 4  → cellstate_binary_profile.csv

  Cell state별 대표 binary profile (majority vote, 0.5 기준):
           ATF3  ATF4  BHLHE40  CEBPB  CEBPD  DBP  DDIT3  E2F1  E2F8  ELF3  ELK3  FOS  FOSB  FOSL1  FOSL2  FOXL1  GATA5  GATA6  HES1  HMGA1  HMGA2  JUN  JUNB  JUND  KLF10  KLF4  KLF5  KLF6  LTF  MAFF  MAFG  MAFK  MYBL2  PTF1A  RBPJL  SOX11  SOX17  SOX18  SOX4  SOX9  SPIB  YBX1
cell_type                                                                                                                                                                                                                                                                            
ADM           1     1        0      0      0    0      1     0     0     0     0    1     1      0      0      0      0      0     1      0      0    1     1     1      0     0     0     1    0     0     0     0  

In [691]:
# ── Step 5: Boolean logic 학습 : CONFIG에서 QMC 방식으로 설정 ───────────────────────────────────
rules_dict, accuracy_dict = step5_learn_boolean_rules(
    binary_sorted, tf_tf_net, CONFIG
)
#10,  1  평균 accuracy  : 0.8286 --> PDAC 비율이 너무 낮음. 6%. MAFF= 0 이 상위 타겟으로 잘 나오긴함. 
#10 , 2  평균 accuracy  : 0.8297 --> PDAC 비율 20%대. 나쁘지 않음. MAFF = 0 3등 YBX1 = 0 1등, 아쉬운건 타겟들 효과가 좀 비슷하다 정도?
#20,  2  평균 accuracy  : 0.8268 --> PDAC 40% , Acinar 4% / 타겟들 가장 괜찮은 것 같음. YBX1은 Acinar은 못늘리지만 MAFF이런앤 늘림. 
#50 , 5  평균 accuracy  : 0.8247 --> 일단 선택. PDAC 비율이 좀 높다?


  Rule learning method: 'qmc'

STEP 5-2: Boolean Rule 학습 (Quine-McCluskey)
  전이 쌍 수: 7771  (window_size=20, window_step=2)
  [   1/42] HMGA1                acc=0.921  on=1  dc=0  regs=['HMGA2', 'YBX1']
  [  11/42] JUNB                 acc=0.899  on=36  dc=77  regs=['ATF4', 'MAFF', 'FOS', 'JUN', 'ATF3', 'FOSB', 'SOX4']
  [  31/42] SOX18                acc=0.931  on=1  dc=0  regs=['ELK3']
  [  41/42] MYBL2                acc=0.879  on=3  dc=0  regs=['E2F1', 'E2F8', 'KLF6']

  평균 accuracy  : 0.8268
  중앙값 accuracy: 0.8253

  ── Self-loop / 상수 rule 요약 ──
  [원인 1] in-edge 없음      : 2 TFs
           → ['PTF1A', 'YBX1']
  [원인 2] 상수 rule (always 0): 9 TFs
           → ['GATA6', 'DBP', 'FOXL1', 'LTF', 'SOX17', 'CEBPD', 'FOSL1', 'SOX11', 'SPIB']
  ※ 원인 2 TF들은 binarization 후에도 거의 꺼진 상태입니다.
     auc_celltype_min_mean 또는 rss_threshold를 높이면 줄어듭니다.


In [692]:
# ── Step 6: .bnet  파일저장 ────────────────────────────────────────────────────
bnet_path = step6_export_network(rules_dict, tf_tf_net, accuracy_dict, CONFIG)


STEP 6: 네트워크 파일 저장
  .bnet 저장: ./PDAC_boolean_output/pdac_boolean_network.bnet
  총 42 TF 규칙
  SBML 변환 스킵 (libsbml 미설치). pip install python-libsbml 후 재실행


In [693]:
# ── Step 6-2: 고립된 상수 및 self-loop 노드 제거 ────────────────────────────────────
# auto_remove=True   : 고립 상수 노드 자동 제거
# manual_remove=[..] : 추가로 직접 제거할 TF 리스트 (빈 리스트면 수동 제거 없음)
MANUAL_REMOVE = [
    # 여기에 제거할 TF명 입력
    # 예) "HMGA2", "DDIT3",
]

rules_dict_after, tf_tf_net_after, accuracy_dict_after = step6_2_remove_nodes(
    rules_dict, tf_tf_net, accuracy_dict,
    auto_remove   = True, # True면 고립된 상수 노드 및 self-loop 자동 탐색 및 제거
    manual_remove = MANUAL_REMOVE,
)

## 상수 노드 FOSL1 = 0 을 network logic 자체에 반영하고 노드는 제거했음! -260429


STEP 6-2: 노드 제거
  시작: 42 nodes, 229 edges

  [자동] 고립 상수 노드: 8개 제거
    상수 0    : 7개  ['DBP', 'FOXL1', 'GATA6', 'LTF', 'SOX11', 'SOX17', 'SPIB']
    상수 1    : 0개  []
    self-loop : 1개  ['PTF1A']

  결과: 34 nodes, 208 edges  (총 8개 제거)
  → ./PDAC_boolean_output/step6_2_removed_nodes.csv 저장


In [694]:
# ── Step 6: .bnet 재저장 ────────────────────────────────────────────────────
bnet_path = step6_export_network(rules_dict_after, tf_tf_net_after, accuracy_dict_after, CONFIG)


STEP 6: 네트워크 파일 저장
  .bnet 저장: ./PDAC_boolean_output/pdac_boolean_network.bnet
  총 34 TF 규칙
  SBML 변환 스킵 (libsbml 미설치). pip install python-libsbml 후 재실행


In [663]:
# ── Step 7: Attractor to UMAP mapping ────────────────────────────────────────────────────

ATTRACTOR_CSV = "/home/yeohs0212/MM/IQcell/pdac_boolean_output_2/normal_attr_df.csv"  # ← R 결과 경로

attractor_mapping = step7_attractor_cell_mapping(
    attractor_df_path = ATTRACTOR_CSV,
    cell_binary       = cell_binary,      # step4_2 결과
    adata             = adata,
    config            = CONFIG,
    dist_threshold    = None,   # float 설정 시 초과 attractor는 UMAP에 미표시
)


STEP 7: Attractor → Cell 매핑 + UMAP 시각화 (Euclidean distance)
  Attractor 수   : 115
  Attractor nodes: 34

  Cyclic attractor: 98개  (소수값 그대로 Euclidean 계산, 이진화 없음)
    Attr 17: {'ATF3': 0.5, 'ATF4': 0.5, 'BHLHE40': 0.5, 'CEBPB': 0.5, 'DDIT3': 0.5, 'FOS': 0.5, 'FOSL2': 0.5, 'HES1': 0.5, 'JUNB': 0.5, 'JUND': 0.5, 'KLF4': 0.5, 'MAFG': 0.5}
    Attr 18: {'ATF4': 0.5, 'E2F1': 0.5, 'ELF3': 0.5, 'FOS': 0.5, 'FOSB': 0.5, 'GATA5': 0.5, 'HES1': 0.5, 'JUN': 0.5, 'JUNB': 0.5, 'JUND': 0.5, 'KLF4': 0.5, 'KLF5': 0.5, 'KLF6': 0.5, 'MAFF': 0.5, 'MAFK': 0.5, 'MYBL2': 0.5, 'RBPJL': 0.5, 'SOX9': 0.5}
    Attr 19: {'E2F1': 0.5, 'MYBL2': 0.5}
    Attr 20: {'ELF3': 0.5, 'GATA5': 0.5, 'MAFG': 0.5, 'MAFK': 0.5}
    Attr 21: {'ATF4': 0.5, 'DDIT3': 0.5, 'FOSB': 0.5, 'JUN': 0.5, 'JUNB': 0.5, 'MAFG': 0.5, 'RBPJL': 0.5}
    Attr 22: {'E2F1': 0.5, 'MYBL2': 0.5}
    Attr 23: {'ATF4': 0.5, 'DDIT3': 0.5, 'ELF3': 0.5, 'FOSB': 0.5, 'GATA5': 0.5, 'JUN': 0.5, 'JUNB': 0.5, 'MAFG': 0.5, 'RBPJL': 0.5}
    Attr 24: {'ELF3': 0.5,


  매핑 결과:
    Attr    0  →  cell 197188186753451_DACC963_mKate_plus  (euclid=1.000, norm=0.172)  [PDAC]
    Attr    1  →  cell 230816733908315_DACD394_Kate_plus  (euclid=0.000, norm=0.000)  [ADM]
    Attr    2  →  cell 197264065850787_DAC_B530-Kate+  (euclid=1.000, norm=0.172)  [ADM]
    Attr    3  →  cell 236082083579750_Ag-PDAC-PT-Kate  (euclid=0.000, norm=0.000)  [PDAC]
    Attr    4  →  cell 239596337248677_DACD511_Kate_plus  (euclid=0.000, norm=0.000)  [Acinar]
    Attr    5  →  cell 239596318902627_DAC_B530-Kate+  (euclid=1.414, norm=0.242)  [ADM]
    Attr    6  →  cell 239596318902627_DAC_B530-Kate+  (euclid=1.732, norm=0.297)  [ADM]
    Attr    7  →  cell 191026802350382_DACD407_Kate_plus  (euclid=0.000, norm=0.000)  [Bridge]
    Attr    8  →  cell 240617687992757_DACC963_mKate_plus  (euclid=0.000, norm=0.000)  [PDAC]
    Attr    9  →  cell 134033672619957_DAC_DI143_Epi  (euclid=1.000, norm=0.172)  [ADM]
    Attr   10  →  cell 157072030816557_DACD550_kate_plus  (euclid=0.000, n


  → attractor_umap.pdf/png 저장
  → attractor_euclidean_dist.pdf/png 저장
./pdac_boolean_output_2/nominal_attractor_celltype.csv에 nominal 정보 저장

Attractor Mapping Summary by Cell Type
                  attractor_count  total_basin_size  size_percentage
mapped_cell_type                                                    
ADM                            26            130596        50.342890
PDAC                           56            107340        41.378034
Acinar                         17             11513         4.438097
Bridge                         16              9964         3.840979


In [664]:
# ── Step 8: 시각화 ────────────────────────────────────────────────────────
step8_visualize_network(G, tf_tf_net_after)


STEP 8: 네트워크 시각화
  → 네트워크 시각화 저장: tf_network_visualization.pdf/png


In [ ]:
#all combination simulation 이후 PDAC 비율 가장 많이 줄이는거로 확인 or Acinar 가장 많이 만든거로 확인. 

In [665]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 7-8: Perturbation별 Cell Type Basin 비율 계산
# ════════════════════════════════════════════════════════════════════════════
def step7_8_perturbation_basin_summary(perturb_dir, cell_binary, adata, config):
    """
    perturbation_results/ 디렉토리의 모든 CSV를 읽어
    각 perturbation에서 cell type별 basin size 비율을 계산.

    각 perturbation CSV 형식:
      rows = attractors, columns = TF nodes + "size"
      (step7_attractor_cell_mapping과 동일한 형식)

    반환 DataFrame:
      rows    = cell type
      columns = perturbation 이름 (perturb_ATF3_fix_0 등)
      values  = basin size 비율 (%)

    출력 파일
    ----------
    perturbation_basin_summary.csv   : 비율 DataFrame
    perturbation_basin_heatmap.pdf   : 히트맵 시각화
    """
    import glob
    import os

    print("\n" + "="*70)
    print("STEP 7-8: Perturbation별 Cell Type Basin 비율")
    print("="*70)

    # ── CSV 파일 목록 수집 ────────────────────────────────────────────────────
    csv_files = sorted(glob.glob(os.path.join(perturb_dir, "perturb_*.csv")))
    print(f"  발견된 perturbation CSV: {len(csv_files)}개")
    if not csv_files:
        raise FileNotFoundError(f"perturb_*.csv 파일이 없습니다: {perturb_dir}")

    # ── 공통 TF 및 cell binary 준비 ──────────────────────────────────────────
    meta_cols  = {"cell_type", "pseudotime"}
    tf_in_cell = [c for c in cell_binary.columns if c not in meta_cols]

    # cell_type 정보 확인
    if "cell_type" not in cell_binary.columns:
        raise ValueError("cell_binary에 'cell_type' 컬럼이 없습니다.")
    all_cell_types = sorted(cell_binary["cell_type"].unique())
    print(f"  Cell types: {all_cell_types}")

    # ── 단일 perturbation 처리 함수 ──────────────────────────────────────────
    def _compute_basin_ratio(csv_path):
        """
        하나의 perturbation CSV에서
        step7 방식으로 attractor → cell 매핑 후
        cell type별 basin size 비율(%) 계산.
        """
        attr_df   = pd.read_csv(csv_path, index_col=0)
        size_col  = [c for c in attr_df.columns if c == "Attr_Size"]
        if not size_col:
            return None   # size 컬럼 없으면 스킵
        attr_size = attr_df[size_col[0]].values.astype(float)
        node_cols = [c for c in attr_df.columns if c != "Attr_Size"]

        # 소수값 그대로 유지 (cyclic attractor 처리)
        attr_raw   = attr_df[node_cols].astype(float)
        common_tfs = [tf for tf in node_cols if tf in tf_in_cell]
        if not common_tfs:
            return None

        cell_mat = cell_binary[common_tfs].values.astype(float)
        attr_mat = attr_raw[common_tfs].values.astype(float)
        n_tfs    = len(common_tfs)

        # Euclidean distance: attractor × cell
        attr_expanded = attr_mat[:, np.newaxis, :]
        cell_expanded = cell_mat[np.newaxis, :, :]
        euclid_mat    = np.sqrt(((attr_expanded - cell_expanded) ** 2).sum(axis=2))

        # 각 attractor → 가장 가까운 cell → 해당 cell의 cell_type
        best_cell_idx = euclid_mat.argmin(axis=1)
        mapped_cts    = [
            cell_binary["cell_type"].iloc[i] for i in best_cell_idx
        ]

        # cell type별 basin size 합산 → 비율(%)
        ct_basin = {ct: 0.0 for ct in all_cell_types}
        for ct, sz in zip(mapped_cts, attr_size):
            if ct in ct_basin:
                ct_basin[ct] += sz

        total = sum(ct_basin.values())
        if total == 0:
            return {ct: 0.0 for ct in all_cell_types}

        return {ct: round(v / total * 100, 4) for ct, v in ct_basin.items()}

    # ── 모든 perturbation 순회 ───────────────────────────────────────────────
    results = {}   # {perturb_name: {cell_type: %}}

    for csv_path in csv_files:
        perturb_name = os.path.splitext(os.path.basename(csv_path))[0]
        ratio = _compute_basin_ratio(csv_path)
        if ratio is None:
            print(f"  ⚠ 스킵: {perturb_name} (size 컬럼 없거나 공통 TF 없음)")
            continue
        results[perturb_name] = ratio

    print(f"  처리 완료: {len(results)}개 perturbation")

    # ── DataFrame 구성 ───────────────────────────────────────────────────────
    # rows = cell type, columns = perturbation
    summary_df = pd.DataFrame(results, index=all_cell_types)
    summary_df.index.name   = "cell_type"
    summary_df.columns.name = "perturbation"

    out_csv = os.path.join(OUT, "perturbation_basin_summary.csv")
    summary_df.to_csv(out_csv)
    print(f"\n  Basin 비율 DataFrame: {summary_df.shape}")
    print(f"  → {out_csv} 저장")
    print(f"\n{summary_df.to_string()}")

    # ── 히트맵 시각화 ────────────────────────────────────────────────────────
    n_perturbs  = len(summary_df.columns)
    n_cts       = len(summary_df.index)
    fig_w       = max(12, n_perturbs * 0.35)
    fig_h       = max(3,  n_cts * 0.7)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        summary_df,
        ax       = ax,
        cmap     = "YlOrRd",
        vmin     = 0,
        vmax     = 100,
        annot    = (n_perturbs <= 40),   # 컬럼 40개 이하면 숫자 표시
        fmt      = ".1f",
        annot_kws= {"size": 7},
        linewidths   = 0.3,
        linecolor    = "#eee",
        cbar_kws = {"label": "Basin size (%)","shrink": 0.7},
    )
    ax.set_title("Cell Type Basin Size (%) per Perturbation",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Perturbation", fontsize=10)
    ax.set_ylabel("Cell Type",    fontsize=10)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)
    plt.tight_layout()

    out_pdf = os.path.join(OUT, "perturbation_basin_heatmap.pdf")
    out_png = os.path.join(OUT, "perturbation_basin_heatmap.png")
    plt.savefig(out_pdf, dpi=150, bbox_inches="tight")
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → perturbation_basin_heatmap.pdf/png 저장")

    return summary_df




In [695]:
PERTURB_DIR = OUT + "/perturbation_results/"

perturb_summary = step7_8_perturbation_basin_summary(
    perturb_dir  = PERTURB_DIR,
    cell_binary  = cell_binary,   # step4_2 결과
    adata        = adata,
    config       = CONFIG,
)

SyntaxError: unterminated string literal (detected at line 1) (35873263.py, line 1)

In [667]:
perturb_summary.columns = (
    perturb_summary.columns
    .str.replace(r'^perturb_', '', regex=True)
    .str.replace('_fix_', ' = ')
)

perturb_summary

perturbation,ATF3 = 0,ATF3 = 1,ATF4 = 0,ATF4 = 1,BHLHE40 = 0,BHLHE40 = 1,CEBPB = 0,CEBPB = 1,CEBPD = 0,CEBPD = 1,...,RBPJL = 0,RBPJL = 1,SOX18 = 0,SOX18 = 1,SOX4 = 0,SOX4 = 1,SOX9 = 0,SOX9 = 1,YBX1 = 0,YBX1 = 1
cell_type,,,,,,,,,,,,,,,,,,,,,
ADM,41.8884,57.0412,59.4315,52.1950,49.0836,42.1867,81.7166,50.7741,50.7028,49.2583,...,48.3513,56.9937,50.3563,50.0699,32.2362,56.3981,50.3264,45.9717,91.1427,9.1777
Acinar,19.5609,0.3736,12.1224,0.2920,4.8083,10.4438,11.0558,3.0210,4.3426,2.0492,...,4.3969,4.4557,4.4625,4.4275,21.2547,0.0632,4.4925,4.4670,8.8573,0.0000
Bridge,8.5949,0.1354,24.5178,0.4440,0.9713,9.5644,1.2103,0.6050,4.3761,1.7119,...,6.7691,1.7924,4.4846,5.0036,2.3983,0.5598,4.4256,3.2676,0.0000,8.9378
PDAC,29.9557,42.4498,3.9283,47.0691,45.1368,37.8051,6.0173,45.5999,40.5785,46.9806,...,40.4827,36.7582,40.6966,40.4991,44.1108,42.9789,40.7555,46.2936,0.0000,81.8845


In [673]:
perturb_summary['MAFF = 0'] # PDAC 10.477

cell_type
ADM       42.6617
Acinar    39.1304
Bridge     7.7303
PDAC      10.4777
Name: MAFF = 0, dtype: float64

In [669]:
nominal_attr_df = pd.read_csv(OUT +"/nominal_attractor_celltype.csv")
nominal_attr_df

,mapped_cell_type,attractor_count,total_basin_size,size_percentage
0,ADM,26,130596,50.342890
1,PDAC,56,107340,41.378034
2,Acinar,17,11513,4.438097
3,Bridge,16,9964,3.840979


In [670]:
nominal_attr_df.set_index('mapped_cell_type')['size_percentage']

mapped_cell_type
ADM       50.342890
PDAC      41.378034
Acinar     4.438097
Bridge     3.840979
Name: size_percentage, dtype: float64

In [671]:
# ════════════════════════════════════════════════════════════════════════════
#  Perturbation vs Nominal 비교
# ════════════════════════════════════════════════════════════════════════════
def perturbation_vs_nominal(perturb_summary, nominal_attr_df):
    """
    perturbation별 cell type basin 비율과 nominal(WT) 비율을 비교.

    Parameters
    ----------
    perturb_summary : pd.DataFrame
        step7_8 결과.
        rows = cell type, columns = perturbation name, values = basin %
    nominal_attr_df : pd.DataFrame
        step7 result_df.
        "mapped_cell_type" 컬럼에 cell type,
        "size_percentage"  컬럼에 비율(%) 저장

    반환
    ----
    delta_df   : perturb % - nominal %  (같은 shape as perturb_summary)
    nominal_series : cell type별 nominal 비율 (index = cell type)

    출력 파일
    ----------
    perturbation_delta_summary.csv    : delta DataFrame
    perturbation_delta_heatmap.pdf    : delta 히트맵 (diverging colormap)
    perturbation_comparison.pdf       : nominal 막대 + delta 히트맵 나란히
    """
    print("\n" + "="*70)
    print("STEP 7-9: Perturbation vs Nominal 비교")
    print("="*70)

    # ── Nominal 비율 추출 ─────────────────────────────────────────────────────
    # perturb_summary의 index(cell type 순서)에 맞게 reindex
    nominal_series = (
        nominal_attr_df
        .set_index("mapped_cell_type")["size_percentage"]
        .reindex(perturb_summary.index)   # perturb_summary index 순서 기준
        .fillna(0.0)
    )
    nominal_series.index.name = "cell_type"

    print(f"  Nominal basin 비율 (index 순서 = perturb_summary 기준):")
    for ct, pct in nominal_series.items():
        print(f"    {ct:<25}: {pct:.2f}%")

    # ── Delta 계산: perturb % - nominal % ────────────────────────────────────
    # perturb_summary: rows=cell_type, cols=perturbation
    # nominal_series : index=cell_type (같은 순서 보장됨)
    delta_df = perturb_summary.subtract(nominal_series, axis=0)
    # axis=0: 각 row(cell type)에서 해당 nominal 값을 빼줌

    delta_df.index.name   = "cell_type"
    delta_df.columns.name = "perturbation"

    out_csv = os.path.join(OUT, "perturbation_delta_summary.csv")
    delta_df.to_csv(out_csv)
    print(f"\n  Delta DataFrame: {delta_df.shape}")
    print(f"  → {out_csv} 저장")

    # ── PDAC 기준 컬럼 오름차순 정렬 ─────────────────────────────────────────
    # PDAC row가 있는지 확인 (대소문자 무관하게 탐색)
    pdac_row = next(
        (idx for idx in delta_df.index if "pdac" in idx.lower()), None
    )
    if pdac_row is None:
        print("  ⚠ 'PDAC' cell type 없음 — 정렬 스킵")
        sort_order = delta_df.columns.tolist()
    else:
        # PDAC delta 값 기준 컬럼 오름차순 정렬
        sort_order = (
            delta_df.loc[pdac_row]
            .sort_values(ascending=True)
            .index.tolist()
        )
        print(f"  정렬 기준: '{pdac_row}' row, delta 오름차순")

    delta_df_sorted       = delta_df[sort_order]
    perturb_summary_sorted = perturb_summary[sort_order]

    # CSV 저장 (정렬된 버전)
    delta_df_sorted.to_csv(os.path.join(OUT, "perturbation_delta_summary.csv"))
    print(f"\n  Delta DataFrame: {delta_df_sorted.shape}")
    print(f"  → perturbation_delta_summary.csv 저장")

    # ── 공통 시각화 파라미터 ──────────────────────────────────────────────────
    n_perturbs = len(delta_df_sorted.columns)
    n_cts      = len(delta_df_sorted.index)
    fig_w      = max(12, n_perturbs * 0.35)
    fig_h      = max(3,  n_cts * 0.7)
    abs_max    = delta_df_sorted.abs().max().max()

    # ── 시각화 1: Delta 히트맵 ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        delta_df_sorted,
        ax        = ax,
        cmap      = "RdBu_r",
        center    = 0,
        vmin      = -abs_max,
        vmax      =  abs_max,
        annot     = (n_perturbs <= 40),
        fmt       = ".1f",
        annot_kws = {"size": 7},
        linewidths  = 0.3,
        linecolor   = "#eee",
        cbar_kws  = {"label": "Δ Basin size (% vs Nominal)", "shrink": 0.7},
    )
    ax.set_title(
        f"Δ Basin Size (Perturbation − Nominal) per Cell Type\n"
        f"(columns sorted by '{pdac_row}' Δ, ascending)",
        fontsize=12, fontweight="bold"
    )
    ax.set_xlabel("Perturbation", fontsize=10)
    ax.set_ylabel("Cell Type",    fontsize=10)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "perturbation_delta_heatmap.pdf"),
                dpi=150, bbox_inches="tight")
    plt.savefig(os.path.join(OUT, "perturbation_delta_heatmap.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → perturbation_delta_heatmap.pdf/png 저장")

    # ── 시각화 2: Nominal 막대 + Delta 히트맵 나란히 ─────────────────────────
    fig, axes = plt.subplots(
        1, 2,
        figsize     = (fig_w + 3, fig_h),
        gridspec_kw = {"width_ratios": [1, max(4, n_perturbs * 0.3)],
                       "wspace": 0.05},
    )

    # 좌: Nominal 수평 막대
    ax_bar = axes[0]
    colors = plt.cm.Set2(np.linspace(0, 1, n_cts))
    bars   = ax_bar.barh(
        nominal_series.index[::-1],
        nominal_series.values[::-1],
        color=colors[::-1], edgecolor="black", linewidth=0.6,
    )
    ax_bar.bar_label(bars,
                     labels=[f"{v:.1f}%" for v in nominal_series.values[::-1]],
                     padding=3, fontsize=8)
    ax_bar.set_xlabel("Basin size (%)", fontsize=9)
    ax_bar.set_title("Nominal", fontsize=11, fontweight="bold")
    ax_bar.set_xlim(0, max(nominal_series.max() * 1.3, 1))
    ax_bar.invert_xaxis()
    ax_bar.tick_params(axis="y", labelsize=9)

    # 우: Delta 히트맵 (정렬된 버전)
    ax_hm = axes[1]
    sns.heatmap(
        delta_df_sorted,
        ax          = ax_hm,
        cmap        = "RdBu_r",
        center      = 0,
        vmin        = -abs_max,
        vmax        =  abs_max,
        annot       = (n_perturbs <= 40),
        fmt         = ".1f",
        annot_kws   = {"size": 7},
        linewidths  = 0.3,
        linecolor   = "#eee",
        cbar_kws    = {"label": "Δ Basin (%p)", "shrink": 0.7},
        yticklabels = False,
    )
    ax_hm.set_title(
        f"Δ Basin (Perturbation − Nominal)\n",
        fontsize=11, fontweight="bold"
    )
    ax_hm.set_xlabel("Perturbation", fontsize=10)
    ax_hm.set_ylabel("")
    ax_hm.tick_params(axis="x", rotation=90, labelsize=7)
    plt.savefig(os.path.join(OUT, "perturbation_comparison.pdf"),
                dpi=150, bbox_inches="tight")
    plt.savefig(os.path.join(OUT, "perturbation_comparison.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → perturbation_comparison.pdf/png 저장")

    return delta_df_sorted, nominal_series

In [672]:
delta_df, nominal_series = perturbation_vs_nominal(
    perturb_summary = perturb_summary,    # step7_8 결과
    nominal_attr_df = nominal_attr_df,  # step7   결과 (result_df)
)


STEP 7-9: Perturbation vs Nominal 비교
  Nominal basin 비율 (index 순서 = perturb_summary 기준):
    ADM                      : 50.34%
    Acinar                   : 4.44%
    Bridge                   : 3.84%
    PDAC                     : 41.38%

  Delta DataFrame: (4, 68)
  → ./pdac_boolean_output_2/perturbation_delta_summary.csv 저장
  정렬 기준: 'PDAC' row, delta 오름차순

  Delta DataFrame: (4, 68)
  → perturbation_delta_summary.csv 저장
  → perturbation_delta_heatmap.pdf/png 저장
  → perturbation_comparison.pdf/png 저장
